男性の声の音域アップ版

In [ ]:
import csv
import json
import os
import queue
import sqlite3
import sys
import threading
import time
import wave
from datetime import datetime
import librosa
import numpy as np

# Jupyter Notebookの表示クリア用
try:
    from IPython.display import clear_output
except ImportError:
    clear_output = None

try:
    import sounddevice as sd
except ImportError:
    sd = None

try:
    from vosk import KaldiRecognizer, Model
except ImportError:
    Model = None
    KaldiRecognizer = None

# ==============================================================================
# --- 0. SQLiteデータベース管理クラス (ロック防止対策済み) ---
# ==============================================================================
DB_PATH = "results/karuta_results.db"


class KarutaSessionManager:

    def __init__(self, db_path=DB_PATH):
        self.db_path = db_path
        self.session_id = None
        self.card_order = 0

        # 保存先フォルダとテーブルの自動生成
        os.makedirs(os.path.dirname(self.db_path), exist_ok=True)
        self._init_db()

    def _get_connection(self):
        # timeout=20.0 を指定してデータベースのロック競合を防止
        return sqlite3.connect(self.db_path, timeout=20.0)

    def _init_db(self):
        conn = self._get_connection()
        try:
            cursor = conn.cursor()

            # テストセッションテーブル
            cursor.execute("""
                CREATE TABLE IF NOT EXISTS test_sessions (
                    session_id INTEGER PRIMARY KEY AUTOINCREMENT,
                    session_type TEXT,
                    session_date TEXT,
                    start_datetime TEXT,
                    end_datetime TEXT,
                    reader_name TEXT,
                    gender TEXT,
                    dictionary TEXT,
                    total_cards INTEGER
                )
            """)

            # 認識ログテーブル
            cursor.execute("""
                CREATE TABLE IF NOT EXISTS recognition_logs (
                    log_id INTEGER PRIMARY KEY AUTOINCREMENT,
                    session_id INTEGER,
                    card_order INTEGER,
                    date_only TEXT,
                    datetime TEXT,
                    reader_name TEXT,
                    raw_audio_name TEXT,
                    raw_audio_path TEXT,
                    processed_audio_path TEXT,
                    detail_csv_filename TEXT,
                    detail_csv_path TEXT,
                    final_id TEXT,
                    final_similarity TEXT,
                    recognized_words TEXT,
                    detail_logs_json TEXT,
                    FOREIGN KEY (session_id) REFERENCES test_sessions(session_id)
                )
            """)
            conn.commit()
        finally:
            conn.close()

    def start_session(
        self,
        reader_name="稲葉(inaba)専任読手",
        gender="ー",
        dictionary="ー",
    ):
        now_str = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        date_only = datetime.now().strftime("%Y-%m-%d")

        conn = self._get_connection()
        try:
            cursor = conn.cursor()
            cursor.execute(
                """
                INSERT INTO test_sessions (
                    session_type, session_date, start_datetime, end_datetime, 
                    reader_name, gender, dictionary, total_cards
                ) VALUES (?, ?, ?, ?, ?, ?, ?, ?)
            """,
                (
                    "100首セット",
                    date_only,
                    now_str,
                    now_str,
                    reader_name,
                    gender,
                    dictionary,
                    0,
                ),
            )

            self.session_id = cursor.lastrowid
            self.card_order = 0
            conn.commit()
            return self.session_id
        finally:
            conn.close()

    def log_card(
        self,
        reader_name,
        raw_audio_name,
        raw_audio_path,
        detail_csv_filename,
        detail_csv_path,
        final_id,
        final_similarity,
        recognized_words,
        detail_logs_json="[]",
        processed_audio_path="ー",
    ):
        if not self.session_id:
            return

        self.card_order += 1
        now_str = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        date_only = datetime.now().strftime("%Y-%m-%d")

        conn = self._get_connection()
        try:
            cursor = conn.cursor()
            cursor.execute(
                """
                INSERT INTO recognition_logs (
                    session_id, card_order, date_only, datetime, reader_name,
                    raw_audio_name, raw_audio_path, processed_audio_path,
                    detail_csv_filename, detail_csv_path, final_id,
                    final_similarity, recognized_words, detail_logs_json
                ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            """,
                (
                    self.session_id,
                    self.card_order,
                    date_only,
                    now_str,
                    reader_name,
                    raw_audio_name,
                    raw_audio_path,
                    processed_audio_path,
                    detail_csv_filename,
                    detail_csv_path,
                    final_id,
                    final_similarity,
                    recognized_words,
                    detail_logs_json,
                ),
            )

            cursor.execute(
                """
                UPDATE test_sessions
                SET end_datetime = ?, total_cards = ?
                WHERE session_id = ?
            """,
                (now_str, self.card_order, self.session_id),
            )

            conn.commit()
        finally:
            conn.close()

    def close_session(self):
        if self.session_id:
            now_str = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            final_session_type = (
                "100首セット" if self.card_order >= 100 else "単体/個別試行"
            )

            conn = self._get_connection()
            try:
                cursor = conn.cursor()
                cursor.execute(
                    """
                    UPDATE test_sessions
                    SET session_type = ?, end_datetime = ?, total_cards = ?
                    WHERE session_id = ?
                """,
                    (
                        final_session_type,
                        now_str,
                        self.card_order,
                        self.session_id,
                    ),
                )
                conn.commit()
            finally:
                conn.close()

            print("\n==========================================")
            print(f"[DB] セッション完了: ID={self.session_id}")
            print(f"     判定結果: {final_session_type}")
            print(f"     総認識枚数: {self.card_order} 枚")
            print("==========================================")


# ==============================================================================
# --- 1. 基本設定 ---
# ==============================================================================
DICTIONARIES = {
    "1": {
        "name": "ひらがな版（統一文字数辞書）",
        "path": "data/karuta_hiragana_aligned.csv",
        "type": "aligned_hiragana",
    },
    "2": {
        "name": "カタカナ版（統一文字数辞書）",
        "path": "data/karuta_katakana_aligned.csv",
        "type": "aligned_katakana",
    },
    "3": {
        "name": "決まり字・手動追加版（サマリー辞書）",
        "path": "data/karuta_card_summary_by_kimari_len.csv",
        "type": "summary",
    },
}

RESULTS_CSV = "results/recognition_history.csv"
TEXT_LOG_DIR = "text_logs"
SAMPLERATE = 16000

VOLUME_THRESHOLD = 0.01  # 音量を感知するしきい値
WAIT_SECONDS = 0.8  # 静寂判定時間（秒）
START_SET = 1
MAX_SETS = 102

LIMIT_UONOKU_SECONDS = 2.0  # 上の句認識時間
UONOKU_MIN_SECONDS = 11.0  # 次セット移行ブロック時間


# ==============================================================================
# --- 2. 補助機能（音声・辞書・ログ・表示） ---
# ==============================================================================
def apply_gender_audio_filter(audio_data, gender):
    if gender != "male" or len(audio_data) == 0:
        return audio_data
    try:
        return librosa.effects.pitch_shift(
            y=audio_data, sr=SAMPLERATE, n_steps=5.0
        )
    except Exception:
        return audio_data


def load_karuta_dictionary(dict_info):
    file_path = dict_info["path"]
    dict_type = dict_info["type"]

    if not os.path.exists(file_path):
        print(f"Error: {file_path} が見つかりません。")
        return None

    grammar_set = set()
    tag_map = {}
    card_max_scores = {}

    with open(file_path, "r", encoding="utf-8-sig") as f:
        reader = csv.reader(f)
        header = next(reader, None)

        for row in reader:
            if not row:
                continue
            num = row[0].strip()
            if num in ["0", "00"]:
                continue

            registered_words = []
            if dict_type in ["aligned_hiragana", "aligned_katakana"]:
                if len(row) < 6:
                    continue
                raw_words = row[5].strip()
                if raw_words:
                    registered_words = raw_words.split()
                denominator_seen = set(registered_words)
                try:
                    fixed_max_char_score = int(row[4].strip())
                except ValueError:
                    fixed_max_char_score = sum(
                        len(w) for w in denominator_seen
                    )

            elif dict_type == "summary":
                if len(row) < 14:
                    continue
                for col_idx in range(7, 14):
                    if row[col_idx].strip():
                        registered_words.extend(row[col_idx].strip().split())
                denominator_seen = set(registered_words)
                fixed_max_char_score = sum(len(w) for w in denominator_seen)

            if not denominator_seen:
                continue

            card_max_scores[num] = fixed_max_char_score
            for w in denominator_seen:
                grammar_set.add(w)
                tag_map.setdefault(w, []).append(num)

    return {
        "grammar": list(grammar_set),
        "tag_map": tag_map,
        "max_scores": card_max_scores,
        "dict_name": dict_info["name"],
        "dict_path": file_path,
    }


def print_dashboard(
    scores,
    max_scores,
    partial_text,
    model_name,
    dict_name,
    gender_label,
    current_set,
    total_sets,
    detail_logs,
    status_mode,
    elapsed,
    current_volume,
    pause_time,
):
    if clear_output:
        clear_output(wait=True)
    else:
        os.system("cls" if os.name == "nt" else "clear")

    results = []
    for tid, score in scores.items():
        max_s = max_scores.get(tid, 1)
        sim = (score / max_s) * 100
        results.append({"id": tid, "score": score, "max": max_s, "sim": sim})
    top_5 = sorted(results, key=lambda x: x["sim"], reverse=True)[:5]

    if status_mode == "wait":
        status_label = "【発声待ち】音源（前の札の下の句）の再生を待っています..."
    elif status_mode == "skip":
        if pause_time > 0:
            status_label = f"【下の句 ➔ 静寂検知中】⏱ {pause_time:.1f}秒 / {WAIT_SECONDS}秒で上の句判定へ (加点無効)"
        else:
            status_label = (
                "【前の札の下の句を検出中...】 音圧検出中 (加点無効)"
            )
    elif status_mode == "recognizing":
        rem = max(0.0, LIMIT_UONOKU_SECONDS - elapsed)
        status_label = f"【上の句 認識実行中】 ⏱ 認識残り: {rem:.1f}秒 / {LIMIT_UONOKU_SECONDS}秒"
    else:
        rem_next = max(0.0, UONOKU_MIN_SECONDS - elapsed)
        status_label = f"【認識完了 ➔ 次セット待機中】 ⏱ 次の発声許可まで残り: {rem_next:.1f}秒 (認識停止中)"

    output = [
        "=" * 80,
        f"設定: [Model: {model_name}] [性別: {gender_label}] [進行: {current_set} / {total_sets} セット]",
        f"辞書: [{dict_name}]",
        f"状況: {status_label}",
        f"音量: ［ {current_volume:.4f} ］ (しきい値: {VOLUME_THRESHOLD})",
        "=" * 80,
    ]

    clean_partial = partial_text.replace("\n", " ")
    display_partial = (
        clean_partial[-50:] if len(clean_partial) > 50 else clean_partial
    )
    output.append(f" | 認識中の言葉  | {display_partial.ljust(50)}")
    output.extend([
        "-" * 80,
        f"{'順位':<4}｜{'推定札':<10} | {'加点/満点':<12}｜{'類似度':<8} ｜棒グラフ",
        "-" * 80,
    ])
    for i in range(5):
        if i < len(top_5):
            res = top_5[i]
            bar_count = int(res["sim"] / 5)
            bar = "■" * bar_count + "□" * (20 - bar_count)
            output.append(
                f"{i+1}位 ｜ {res['id']:>3}番      | {res['score']:>3} / {res['max']:>3} pts | {res['sim']:>5.1f}%    ｜|{bar}"
            )
        else:
            output.append(
                f"{i+1}位 ｜ ---番      |    0 / --- pts |    0.0%    ｜|{'□'*20}"
            )

    output.extend(
        ["-" * 80, " 【リアルタイム認識ログ（直近5件）】", "-" * 80]
    )
    if detail_logs:
        for log in detail_logs[-5:][::-1]:
            output.append(
                f" [{log[0]}] 単語: {log[1]:<8} ➔ {log[2]:>3}番に加点 (+{len(log[1])}pts)"
            )
    else:
        output.append(" まだキーワードが認識されていません。")

    output.extend(["=" * 80])
    print("\n".join(output))


def save_to_result_csv(data):
    os.makedirs(os.path.dirname(RESULTS_CSV), exist_ok=True)
    file_exists = os.path.isfile(RESULTS_CSV)
    with open(RESULTS_CSV, "a", encoding="utf-8-sig", newline="") as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow([
                "日時",
                "性別",
                "使用辞書",
                "ファイル名",
                "最終推定ID",
                "最終類似度",
                "認識単語",
            ])
        writer.writerow(data)


def save_to_text_file(
    set_num,
    gender_label,
    dict_name,
    base_name,
    final_id,
    final_sim,
    all_words,
    detail_logs,
):
    os.makedirs(TEXT_LOG_DIR, exist_ok=True)
    txt_path = os.path.join(TEXT_LOG_DIR, f"{base_name}.txt")
    with open(txt_path, "w", encoding="utf-8") as f:
        f.write(f"=== 第 {set_num} セット 解析結果 (2秒判定モデル) ===\n")
        f.write(f"話者性別: {gender_label}\n")
        f.write(f"使用辞書: {dict_name}\n")
        f.write(f"日時: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"識別ファイルベース名: {base_name}\n")
        f.write(f"最終推定札ID: {final_id}\n")
        f.write(f"最終類似度: {final_sim:.1f}%\n")
        f.write(f"認識に成功した全単語群: {' '.join(all_words)}\n")
        f.write("\n--- 時系列詳細加点ログ ---\n")
        for log in detail_logs:
            f.write(
                f"[{log[0]}] 単語: {log[1]:<8} ➔ {log[2]}番に加点 (+{len(log[1])}pts) | 現在累計: {log[3]}/{log[4]} ({log[5]})\n"
            )


# ==============================================================================
# --- 3. メイン認識関数 ---
# ==============================================================================
def run_realtime_karuta():
    print("\n==========================================")
    print("【読手名を入力してください】")
    reader_name = (
        input("読手名 [デフォルト: 稲葉(inaba)専任読手]: ").strip()
        or "稲葉(inaba)専任読手"
    )

    print("\n【話者の性別を選択してください】")
    print(" 1: 女性 (標準)")
    print(" 2: 男性 (ピッチシフト高音化変換)")
    gender_choice = input("選択 (1-2) -> ").strip()
    gender = "male" if gender_choice == "2" else "female"
    gender_label = "男性 (高音ピッチ化)" if gender == "male" else "女性 (標準)"

    print("\n【使用する辞書を選択してください】")
    for key, val in DICTIONARIES.items():
        print(f" {key}: {val['name']}")
    dict_choice = input("選択 (1-3) -> ").strip()

    if dict_choice not in DICTIONARIES:
        print("無効な選択です。")
        return

    selected_dict_info = DICTIONARIES[dict_choice]

    print("\n【使用モデルを選択してください】")
    model_choice = input(
        "1: Standard (精度重視) / 2: Small (速度重視) -> "
    ).strip()
    model_name = "model_standard" if model_choice == "1" else "model_small"

    sys_data = load_karuta_dictionary(selected_dict_info)
    if not sys_data:
        return

    grammar = sys_data["grammar"]
    tag_map = sys_data["tag_map"]
    max_scores = sys_data["max_scores"]
    dict_display_name = sys_data["dict_name"]

    print(f"\n--- Vosk モデル読み込み中 ({model_name}) ---")
    model = Model(model_name)
    rec = KaldiRecognizer(
        model, SAMPLERATE, json.dumps(grammar, ensure_ascii=False)
    )

    # ★ DBセッションの開始
    db = KarutaSessionManager()
    db.start_session(
        reader_name=reader_name,
        gender=gender_label,
        dictionary=dict_display_name,
    )

    q = queue.Queue()

    def callback(indata, frames, time_info, status):
        q.put(bytes(indata))

    state = {
        "current_set": START_SET,
        "scores": {},
        "seen_words": {},
        "all_words": [],
        "detail_logs": [],
        "result_text": "",
        "current_volume": 0.0,
        "is_speaking": False,
        "silence_start_time": None,
        "shimonoku_finished": False,
        "uonoku_start_time": None,
        "status_mode": "wait",
        "elapsed": 0.0,
        "pause_time": 0.0,
        "is_running": True,
        "recorded_raw_frames": [],
        "recorded_processed_frames": [],
    }

    def dashboard_refresh_worker():
        while state["is_running"]:
            if not state["shimonoku_finished"]:
                if state["is_speaking"]:
                    state["status_mode"] = "skip"
                    if state["silence_start_time"] is not None:
                        state["pause_time"] = (
                            time.time() - state["silence_start_time"]
                        )
                    else:
                        state["pause_time"] = 0.0
                else:
                    state["status_mode"] = "wait"
                    state["pause_time"] = 0.0
                state["elapsed"] = 0.0
            else:
                if state["uonoku_start_time"] is not None:
                    state["elapsed"] = (
                        time.time() - state["uonoku_start_time"]
                    )
                state["pause_time"] = 0.0
                if state["elapsed"] <= LIMIT_UONOKU_SECONDS:
                    state["status_mode"] = "recognizing"
                else:
                    state["status_mode"] = "waiting_next"

            print_dashboard(
                state["scores"],
                max_scores,
                state["result_text"],
                model_name,
                dict_display_name,
                gender_label,
                state["current_set"],
                MAX_SETS,
                state["detail_logs"],
                state["status_mode"],
                state["elapsed"],
                state["current_volume"],
                state["pause_time"],
            )
            time.sleep(0.1)

    refresh_thread = threading.Thread(
        target=dashboard_refresh_worker, daemon=True
    )
    refresh_thread.start()

    try:
        with sd.RawInputStream(
            samplerate=SAMPLERATE,
            blocksize=1600,
            channels=1,
            dtype="float32",
            callback=callback,
        ):
            while state["current_set"] <= MAX_SETS and state["is_running"]:
                data = q.get()
                raw_audio_data = np.frombuffer(data, dtype=np.float32)
                if len(raw_audio_data) == 0:
                    continue

                # 生音声データの保持
                raw_bytes = raw_audio_data.tobytes()
                state["recorded_raw_frames"].append(raw_bytes)

                # 音声変換および変換後データの保持
                processed_audio_data = apply_gender_audio_filter(
                    raw_audio_data, gender
                )
                processed_bytes = processed_audio_data.astype(
                    np.float32
                ).tobytes()
                state["recorded_processed_frames"].append(processed_bytes)

                state["current_volume"] = np.sqrt(
                    np.mean(processed_audio_data**2)
                )

                if not state["shimonoku_finished"]:
                    if state["current_volume"] > VOLUME_THRESHOLD:
                        state["is_speaking"] = True
                        state["silence_start_time"] = None
                    else:
                        if state["is_speaking"]:
                            if state["silence_start_time"] is None:
                                state["silence_start_time"] = time.time()
                            elif (
                                time.time() - state["silence_start_time"]
                                >= WAIT_SECONDS
                            ):
                                state["is_speaking"] = False
                                state["shimonoku_finished"] = True
                                state["uonoku_start_time"] = time.time()
                                state["silence_start_time"] = None
                                rec.Reset()
                                state["result_text"] = ""
                else:
                    elapsed_uonoku = (
                        time.time() - state["uonoku_start_time"]
                    )

                    if state["current_volume"] > VOLUME_THRESHOLD:
                        if elapsed_uonoku > UONOKU_MIN_SECONDS:
                            final_id, final_sim = "---", 0.0
                            if state["scores"]:
                                final_id = max(
                                    state["scores"],
                                    key=lambda k: state["scores"][k]
                                    / max_scores.get(k, 1),
                                )
                                final_sim = (
                                    state["scores"][final_id]
                                    / max_scores.get(final_id, 1)
                                ) * 100

                            dt = datetime.now()
                            dt_str = dt.strftime("%Y%m%d_%H%M%S")
                            base_name = f"set_{state['current_set']:03d}_{dt_str}"

                            full_path_wav = os.path.join(
                                "recordings", f"{base_name}.wav"
                            )
                            full_path_processed_wav = os.path.join(
                                "recordings_processed", f"{base_name}_processed.wav"
                            )
                            full_path_csv = os.path.join(
                                "results", f"{base_name}.csv"
                            )

                            os.makedirs("recordings", exist_ok=True)
                            os.makedirs("recordings_processed", exist_ok=True)
                            os.makedirs("results", exist_ok=True)

                            # 1. 生音声WAVの書き出し
                            with wave.open(full_path_wav, "wb") as wf:
                                wf.setnchannels(1)
                                wf.setsampwidth(2)
                                wf.setframerate(SAMPLERATE)
                                all_raw_audio = np.frombuffer(
                                    b"".join(state["recorded_raw_frames"]),
                                    dtype=np.float32,
                                )
                                all_raw_audio_int16 = (all_raw_audio * 32768.0).astype(
                                    np.int16
                                ).tobytes()
                                wf.writeframes(all_raw_audio_int16)

                            # 2. 変換後音声WAVの書き出し
                            with wave.open(full_path_processed_wav, "wb") as wf:
                                wf.setnchannels(1)
                                wf.setsampwidth(2)
                                wf.setframerate(SAMPLERATE)
                                all_processed_audio = np.frombuffer(
                                    b"".join(state["recorded_processed_frames"]),
                                    dtype=np.float32,
                                )
                                all_processed_audio_int16 = (all_processed_audio * 32768.0).astype(
                                    np.int16
                                ).tobytes()
                                wf.writeframes(all_processed_audio_int16)

                            with open(
                                full_path_csv,
                                "w",
                                encoding="utf-8-sig",
                                newline="",
                            ) as f:
                                writer = csv.writer(f)
                                writer.writerow([
                                    "時刻",
                                    "認識単語",
                                    "加点札ID",
                                    "現在スコア",
                                    "満点スコア",
                                    "類似度",
                                ])
                                writer.writerows(state["detail_logs"])

                            save_to_result_csv([
                                dt.strftime("%Y-%m-%d %H:%M:%S"),
                                gender_label,
                                dict_display_name,
                                f"{base_name}.wav",
                                final_id,
                                f"{final_sim:.1f}%",
                                " ".join(state["all_words"]),
                            ])

                            save_to_text_file(
                                state["current_set"],
                                gender_label,
                                dict_display_name,
                                base_name,
                                final_id,
                                final_sim,
                                state["all_words"],
                                state["detail_logs"],
                            )

                            # ★ DBへの記録呼び出し (変換後音声パスを追加設定)
                            db.log_card(
                                reader_name=reader_name,
                                raw_audio_name=f"{base_name}.wav",
                                raw_audio_path=full_path_wav,
                                processed_audio_path=full_path_processed_wav,
                                detail_csv_filename=f"{base_name}.csv",
                                detail_csv_path=full_path_csv,
                                final_id=final_id,
                                final_similarity=f"{final_sim:.1f}%",
                                recognized_words=" ".join(state["all_words"]),
                                detail_logs_json=json.dumps(
                                    state["detail_logs"], ensure_ascii=False
                                ),
                            )

                            state["current_set"] += 1
                            state["scores"] = {}
                            state["seen_words"] = {}
                            state["all_words"] = []
                            state["detail_logs"] = []
                            state["result_text"] = ""
                            state["is_speaking"] = True
                            state["silence_start_time"] = None
                            state["shimonoku_finished"] = False
                            state["uonoku_start_time"] = None
                            state["recorded_raw_frames"] = [raw_bytes]
                            state["recorded_processed_frames"] = [processed_bytes]

                            rec.Reset()
                            if state["current_set"] > MAX_SETS:
                                break
                            continue

                if (
                    state["shimonoku_finished"]
                    and state["uonoku_start_time"] is not None
                ):
                    elapsed_uonoku = (
                        time.time() - state["uonoku_start_time"]
                    )
                    if elapsed_uonoku <= LIMIT_UONOKU_SECONDS:
                        int16_data = (processed_audio_data * 32768.0).astype(
                            np.int16
                        ).tobytes()
                        if not rec.AcceptWaveform(int16_data):
                            state["result_text"] = json.loads(
                                rec.PartialResult()
                            ).get("partial", "")
                        else:
                            state["result_text"] = json.loads(
                                rec.Result()
                            ).get("text", "")

                        if state["result_text"]:
                            new_words = state["result_text"].split()
                            for word in new_words:
                                if word in tag_map:
                                    for tid in tag_map[word]:
                                        state["seen_words"].setdefault(
                                            tid, set()
                                        )
                                        if (
                                            word
                                            not in state["seen_words"][tid]
                                        ):
                                            state["scores"][tid] = state[
                                                "scores"
                                            ].get(tid, 0) + len(word)
                                            state["seen_words"][tid].add(
                                                word
                                            )
                                            if word not in state["all_words"]:
                                                state["all_words"].append(
                                                    word
                                                )

                                            current_sim = (
                                                state["scores"][tid]
                                                / max_scores.get(tid, 1)
                                            ) * 100
                                            state["detail_logs"].append([
                                                datetime.now().strftime(
                                                    "%H:%M:%S.%f"
                                                )[:-3],
                                                word,
                                                tid,
                                                state["scores"][tid],
                                                max_scores.get(tid),
                                                f"{current_sim:.1f}%",
                                            ])

    except KeyboardInterrupt:
        print("\nユーザーによって処理が中断されました。")
    finally:
        state["is_running"] = False
        # ★ DBのクローズ（枚数に応じて100首セット/個別試行を判定）
        db.close_session()


# ==============================================================================
# --- 4. 実行部 ---
# ==============================================================================
run_realtime_karuta()

In [ ]:
import csv
import json
import os
import queue
import sqlite3
import sys
import threading
import time
import wave
from datetime import datetime

from IPython.display import clear_output
import numpy as np
import sounddevice as sd

# ピッチ変換ライブラリ (scipy)
from scipy.signal import resample
from vosk import KaldiRecognizer, Model

# ==============================================================================
# --- 1. 定数・辞書定義・初期設定 ---
# ==============================================================================

DICTIONARIES = {
    "1": {
        "name": "ひらがな版（統一文字数辞書）",
        "path": "data/karuta_hiragana_aligned.csv",
        "type": "aligned_hiragana",
    },
    "2": {
        "name": "カタカナ版（統一文字数辞書）",
        "path": "data/karuta_katakana_aligned.csv",
        "type": "aligned_katakana",
    },
    "3": {
        "name": "決まり字・手動追加版（サマリー辞書）",
        "path": "data/karuta_card_summary_by_kimari_len.csv",
        "type": "summary",
    },
}

RESULTS_CSV = "results/recognition_history.csv"
TEXT_LOG_DIR = "text_logs"
DB_PATH = "data/karuta_results.db"
SAMPLERATE = 16000

# 音量・時間制御パラメタ
VOLUME_THRESHOLD = 0.01  # 音声を感知するしきい値
WAIT_SECONDS = 0.8  # 下の句終了から上の句移行までの静寂時間
START_SET = 1  # 開始セット
MAX_SETS = 102  # 最大セット数

LIMIT_UONOKU_SECONDS = 2.0  # 上の句加点処理を実行する秒数
UONOKU_MIN_SECONDS = 11.0  # 次セットへの移行ブロック時間


# ==============================================================================
# --- 2. データベース制御 (SQLite) ---
# ==============================================================================
def init_db():
    """DBテーブルの初期化およびカラム追加"""
    os.makedirs(os.path.dirname(DB_PATH), exist_ok=True)
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    # メインのセット結果テーブル
    cursor.execute(
        """
    CREATE TABLE IF NOT EXISTS recognition_sets (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        timestamp TEXT,
        reader_name TEXT,
        gender TEXT,
        dict_name TEXT,
        set_num INTEGER,
        wav_filename TEXT,
        final_id TEXT,
        final_similarity REAL,
        recognized_words TEXT
    )
    """
    )

    # 既存DBへのマイグレーション: 変換後音声ファイル名用のカラムがない場合は追加
    cursor.execute("PRAGMA table_info(recognition_sets)")
    columns = [column[1] for column in cursor.fetchall()]
    if "converted_wav_filename" not in columns:
        cursor.execute(
            "ALTER TABLE recognition_sets ADD COLUMN converted_wav_filename TEXT"
        )

    # 詳細加点ログテーブル
    cursor.execute(
        """
    CREATE TABLE IF NOT EXISTS recognition_details (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        set_id INTEGER,
        log_time TEXT,
        word TEXT,
        card_id TEXT,
        current_score INTEGER,
        max_score INTEGER,
        similarity REAL,
        FOREIGN KEY (set_id) REFERENCES recognition_sets (id)
    )
    """
    )
    conn.commit()
    conn.close()


def save_to_db(
    timestamp,
    reader_name,
    gender,
    dict_name,
    set_num,
    wav_filename,
    converted_wav_filename,
    final_id,
    final_sim,
    all_words,
    detail_logs,
):
    """結果をSQLiteにDB保存"""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    cursor.execute(
        """
    INSERT INTO recognition_sets (
        timestamp, reader_name, gender, dict_name, set_num, wav_filename, converted_wav_filename, final_id, final_similarity, recognized_words
    ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """,
        (
            timestamp,
            reader_name,
            gender,
            dict_name,
            set_num,
            wav_filename,
            converted_wav_filename,
            final_id,
            final_sim,
            " ".join(all_words),
        ),
    )

    set_id = cursor.lastrowid

    # 詳細ログの保存
    for log in detail_logs:
        sim_val = float(log[5].replace("%", ""))
        cursor.execute(
            """
        INSERT INTO recognition_details (
            set_id, log_time, word, card_id, current_score, max_score, similarity
        ) VALUES (?, ?, ?, ?, ?, ?, ?)
        """,
            (set_id, log[0], log[1], log[2], log[3], log[4], sim_val),
        )

    conn.commit()
    conn.close()


# ==============================================================================
# --- 3. 音声ピッチ変換ロジック ---
# ==============================================================================
def pitch_shift(audio_data, semitones=4.0):
    """簡易リサンプルによるピッチシフト処理"""
    if semitones == 0:
        return audio_data

    factor = 2.0 ** (semitones / 12.0)
    orig_len = len(audio_data)

    resampled = resample(audio_data, int(orig_len / factor))

    if len(resampled) < orig_len:
        resampled = np.pad(resampled, (0, orig_len - len(resampled)), "constant")
    else:
        resampled = resampled[:orig_len]

    return resampled.astype(np.float32)


# ==============================================================================
# --- 4. 辞書読み込み & 初期化 ---
# ==============================================================================

def prepare_karuta_system(dict_info):
    file_path = dict_info["path"]
    dict_type = dict_info["type"]

    if not os.path.exists(file_path):
        print(f"Error: {file_path} が見つかりません。パスを確認してください。")
        return None

    grammar_set = set()
    tag_map = {}
    card_max_scores = {}

    with open(file_path, "r", encoding="utf-8-sig") as f:
        reader = csv.reader(f)
        header = next(reader, None)

        for row in reader:
            if not row:
                continue

            num = row[0].strip()
            if num in ["0", "00", ""]:
                continue

            registered_words = []

            # --- 1 & 2. ひらがな・カタカナ版辞書 ---
            if dict_type in ["aligned_hiragana", "aligned_katakana"]:
                if len(row) < 6:
                    continue
                raw_words = row[5].strip()
                if raw_words:
                    registered_words = raw_words.split()

                denominator_seen = set(registered_words)
                try:
                    fixed_max_char_score = int(row[4].strip())
                except ValueError:
                    fixed_max_char_score = sum(
                        len(w) for w in denominator_seen
                    )

            # --- 3. サマリー辞書 ---
            elif dict_type == "summary":
                if len(row) < 14:
                    continue
                # 7列目〜13列目 (インデックス 7〜13) の単語をループで回収
                for col_idx in range(7, 14):
                    if row[col_idx].strip():
                        registered_words.extend(row[col_idx].strip().split())

                denominator_seen = set(registered_words)
                fixed_max_char_score = sum(len(w) for w in denominator_seen)

            else:
                continue

            # 単語が何も登録されていない行はスキップ
            if not registered_words:
                continue

            card_max_scores[num] = fixed_max_char_score

            for w in denominator_seen:
                grammar_set.add(w)
                tag_map.setdefault(w, []).append(num)

    print(f"\n[システム初期化完了 - {dict_info['name']}]")
    print(f" ➔ 対象ファイル: {file_path}")
    print(f" ➔ Vosk登録総語数: {len(grammar_set)}語")
    print(f" ➔ 登録札数: {len(card_max_scores)}枚")

    return {
        "grammar": list(grammar_set),
        "tag_map": tag_map,
        "max_scores": card_max_scores,
    }

# ==============================================================================
# --- 5. ダッシュボード表示 & ログ保存 ---
# ==============================================================================
def print_dashboard(
    scores,
    max_scores,
    partial_text,
    model_name,
    current_set,
    total_sets,
    detail_logs,
    status_mode,
    elapsed,
    current_volume,
    pause_time,
    reader_name,
    gender,
    dict_name,
):
    clear_output(wait=True)
    results = []
    for tid, score in scores.items():
        max_s = max_scores.get(tid, 1)
        sim = (score / max_s) * 100
        results.append({"id": tid, "score": score, "max": max_s, "sim": sim})
    top_5 = sorted(results, key=lambda x: x["sim"], reverse=True)[:5]

    if status_mode == "wait":
        status_label = "【発声待ち】音源（前の札の下の句）の再生を待っています..."
    elif status_mode == "skip":
        if pause_time > 0:
            status_label = f"【下の句 ➔ 静寂検知中】⏱ {pause_time:.1f}秒 / {WAIT_SECONDS}秒で上の句判定へ (加点無効)"
        else:
            status_label = "【前の札の下の句を検出中...】 音圧検出中 (加点無効)"
    elif status_mode == "recognizing":
        rem = max(0.0, LIMIT_UONOKU_SECONDS - elapsed)
        status_label = f"【上の句 認識実行中】 ⏱ 認識残り: {rem:.1f}秒 / {LIMIT_UONOKU_SECONDS}秒"
    else:
        rem_next = max(0.0, UONOKU_MIN_SECONDS - elapsed)
        status_label = f"【認識完了 ➔ 次セット待機中】 ⏱ 次の発声許可まで残り: {rem_next:.1f}秒 (認識停止中)"

    output = [
        "=" * 80,
        f"読み手: {reader_name} ({gender}) | 辞書: {dict_name}",
        f"設定: [Model: {model_name}] [進行: {current_set} / {total_sets} セット]",
        f"状況: {status_label}",
        f"音量: ［ {current_volume:.4f} ］ (しきい値: {VOLUME_THRESHOLD})",
        "=" * 80,
    ]

    clean_partial = partial_text.replace("\n", " ")
    display_partial = (
        clean_partial[-50:] if len(clean_partial) > 50 else clean_partial
    )
    output.append(f" | 認識中の言葉  | {display_partial.ljust(50)}")
    output.extend(
        [
            "-" * 80,
            f"{'順位':<4}｜{'推定札':<10} | {'加点/満点':<12}｜{'類似度':<8} ｜棒グラフ",
            "-" * 80,
        ]
    )
    for i in range(5):
        if i < len(top_5):
            res = top_5[i]
            bar_count = int(res["sim"] / 5)
            bar = "■" * bar_count + "□" * (20 - bar_count)
            output.append(
                f"{i+1}位 ｜ {res['id']:>3}番      | {res['score']:>3} / {res['max']:>3} pts | {res['sim']:>5.1f}%    ｜|{bar}"
            )
        else:
            output.append(
                f"{i+1}位 ｜ ---番      |    0 / --- pts |    0.0%    ｜|{'□'*20}"
            )

    output.extend(["-" * 80, " 【リアルタイム認識ログ（直近5件）】", "-" * 80])
    if detail_logs:
        recent_logs = detail_logs[-5:][::-1]
        for log in recent_logs:
            output.append(
                f" [{log[0]}] 単語: {log[1]:<8} ➔ {log[2]:>3}番に加点 (+{len(log[1])}pts)"
            )
    else:
        output.append(" まだキーワードが認識されていません。")

    output.extend(["=" * 80])
    print("\n".join(output))


def save_to_result_csv(data):
    os.makedirs(os.path.dirname(RESULTS_CSV), exist_ok=True)
    file_exists = os.path.isfile(RESULTS_CSV)
    with open(RESULTS_CSV, "a", encoding="utf-8-sig", newline="") as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow(
                [
                    "日時",
                    "読み手",
                    "性別",
                    "生音声ファイル名",
                    "変換音声ファイル名",
                    "最終推定ID",
                    "最終類似度",
                    "認識単語",
                ]
            )
        writer.writerow(data)


# ==============================================================================
# --- 6. メイン実行関数 ---
# ==============================================================================
def run_realtime_karuta():
    init_db()  # DBのテーブル初期設定

    print("=========================================")
    print("    かるた音声認識システム 設定")
    print("=========================================")

    # 1. 読み手名の入力
    print("\n【1】 読み手の名前")
    reader_name = input("【読手入力】 読み手の名前を入力してください: ").strip()
    if not reader_name:
        reader_name = "Unknown"

    # 2. 性別の選択
    print("\n【2】 読み手の性別を選択してください:")
    print("  1: 男性 (ピッチ高音化補正: +4半音)")
    print("  2: 女性 (ピッチ変更なし)")
    g_choice = input("【性別選択】 番号を入力してください (1 or 2): ").strip()

    if g_choice == "1":
        gender = "男性"
        pitch_shift_semitones = 4.0
    else:
        gender = "女性"
        pitch_shift_semitones = 0.0

    # 3. 辞書の選択
    print("\n【3】 使用する辞書を選択してください:")
    for k, v in DICTIONARIES.items():
        print(f"  {k}: {v['name']}")
    d_choice = input("【辞書選択】 番号を入力してください (1〜3): ").strip()
    selected_dict = DICTIONARIES.get(d_choice, DICTIONARIES["1"])

    # 4. モデル選択
    print("\n【4】 使用モデルを選択してください:")
    print("  1: Standard (精度重視) / 2: Small (速度重視)")
    m_choice = input("【モデル選択】 番号を入力してください (1 or 2): ").strip()
    model_name = "model_standard" if m_choice == "1" else "model_small"

    # システム・辞書準備
    sys_data = prepare_karuta_system(selected_dict)
    if not sys_data:
        return

    grammar = sys_data["grammar"]
    tag_map = sys_data["tag_map"]
    max_scores = sys_data["max_scores"]

    print(f"\n--- Vosk モデル読み込み中 ({model_name}) ---")
    model = Model(model_name)
    rec = KaldiRecognizer(model, SAMPLERATE, json.dumps(grammar, ensure_ascii=False))

    q = queue.Queue()

    def callback(indata, frames, time_info, status):
        q.put(bytes(indata))

    state = {
        "current_set": START_SET,
        "scores": {},
        "seen_words": {},
        "all_words": [],
        "detail_logs": [],
        "result_text": "",
        "current_volume": 0.0,
        "is_speaking": False,
        "silence_start_time": None,
        "shimonoku_finished": False,
        "uonoku_start_time": None,
        "status_mode": "wait",
        "elapsed": 0.0,
        "pause_time": 0.0,
        "is_running": True,
        "recorded_frames": [],
        "converted_recorded_frames": [],  # ピッチ変換後音声保持用
    }

    def dashboard_refresh_worker():
        while state["is_running"]:
            if not state["shimonoku_finished"]:
                if state["is_speaking"]:
                    state["status_mode"] = "skip"
                    if state["silence_start_time"] is not None:
                        state["pause_time"] = (
                            time.time() - state["silence_start_time"]
                        )
                    else:
                        state["pause_time"] = 0.0
                else:
                    state["status_mode"] = "wait"
                    state["pause_time"] = 0.0
                state["elapsed"] = 0.0
            else:
                if state["uonoku_start_time"] is not None:
                    state["elapsed"] = time.time() - state["uonoku_start_time"]
                state["pause_time"] = 0.0
                if state["elapsed"] <= LIMIT_UONOKU_SECONDS:
                    state["status_mode"] = "recognizing"
                else:
                    state["status_mode"] = "waiting_next"

            print_dashboard(
                state["scores"],
                max_scores,
                state["result_text"],
                model_name,
                state["current_set"],
                MAX_SETS,
                state["detail_logs"],
                state["status_mode"],
                state["elapsed"],
                state["current_volume"],
                state["pause_time"],
                reader_name,
                gender,
                selected_dict["name"],
            )
            time.sleep(0.1)

    refresh_thread = threading.Thread(target=dashboard_refresh_worker, daemon=True)
    refresh_thread.start()

    print("\n>>> 音声ストリームの受付を開始します。")
    try:
        with sd.RawInputStream(
            samplerate=SAMPLERATE,
            blocksize=1600,
            channels=1,
            dtype="float32",
            callback=callback,
        ):
            while state["current_set"] <= MAX_SETS and state["is_running"]:
                data = q.get()
                state["recorded_frames"].append(data)

                audio_data = np.frombuffer(data, dtype=np.float32)
                if len(audio_data) == 0:
                    continue

                # ピッチ補正（男性声の場合はシフト）
                processed_audio = pitch_shift(audio_data, pitch_shift_semitones)
                
                # 変換済みのデータフレームをバイナリとして保持
                state["converted_recorded_frames"].append(processed_audio.tobytes())

                state["current_volume"] = np.sqrt(np.mean(processed_audio**2))

                # --- 状態管理 ---
                if not state["shimonoku_finished"]:
                    if state["current_volume"] > VOLUME_THRESHOLD:
                        state["is_speaking"] = True
                        state["silence_start_time"] = None
                    else:
                        if state["is_speaking"]:
                            if state["silence_start_time"] is None:
                                state["silence_start_time"] = time.time()
                            elif (
                                time.time() - state["silence_start_time"]
                                >= WAIT_SECONDS
                            ):
                                state["is_speaking"] = False
                                state["shimonoku_finished"] = True
                                state["uonoku_start_time"] = time.time()
                                state["silence_start_time"] = None
                                rec.Reset()
                                state["result_text"] = ""
                else:
                    elapsed_uonoku = time.time() - state["uonoku_start_time"]

                    # 次セットへの移行判定 (11秒以降に次の声を感知)
                    if state["current_volume"] > VOLUME_THRESHOLD:
                        if elapsed_uonoku > UONOKU_MIN_SECONDS:
                            final_id, final_sim = "---", 0.0
                            if state["scores"]:
                                final_id = max(
                                    state["scores"],
                                    key=lambda k: state["scores"][k]
                                    / max_scores.get(k, 1),
                                )
                                final_sim = (
                                    state["scores"][final_id]
                                    / max_scores.get(final_id, 1)
                                ) * 100

                            dt = datetime.now()
                            dt_str = dt.strftime("%Y%m%d_%H%M%S")
                            base_name = f"set_{state['current_set']:03d}_{dt_str}"
                            wav_filename = f"{base_name}.wav"
                            converted_wav_filename = f"{base_name}_converted.wav"

                            # 1-A. 生WAVの保存
                            os.makedirs("recordings", exist_ok=True)
                            full_path_wav = os.path.join("recordings", wav_filename)
                            with wave.open(full_path_wav, "wb") as wf:
                                wf.setnchannels(1)
                                wf.setsampwidth(2)
                                wf.setframerate(SAMPLERATE)
                                all_audio = np.frombuffer(
                                    b"".join(state["recorded_frames"]),
                                    dtype=np.float32,
                                )
                                all_audio_int16 = (
                                    (all_audio * 32768.0)
                                    .astype(np.int16)
                                    .tobytes()
                                )
                                wf.writeframes(all_audio_int16)

                            # 1-B. 変換後WAVの保存
                            os.makedirs("converted_recordings", exist_ok=True)
                            full_path_converted = os.path.join(
                                "converted_recordings", converted_wav_filename
                            )
                            with wave.open(full_path_converted, "wb") as wf:
                                wf.setnchannels(1)
                                wf.setsampwidth(2)
                                wf.setframerate(SAMPLERATE)
                                all_converted_audio = np.frombuffer(
                                    b"".join(state["converted_recorded_frames"]),
                                    dtype=np.float32,
                                )
                                converted_int16 = (
                                    (all_converted_audio * 32768.0)
                                    .astype(np.int16)
                                    .tobytes()
                                )
                                wf.writeframes(converted_int16)

                            # 2. 全体CSV保存
                            save_to_result_csv(
                                [
                                    dt.strftime("%Y-%m-%d %H:%M:%S"),
                                    reader_name,
                                    gender,
                                    wav_filename,
                                    converted_wav_filename,
                                    final_id,
                                    f"{final_sim:.1f}%",
                                    " ".join(state["all_words"]),
                                ]
                            )

                            # 3. DBへ保存
                            save_to_db(
                                dt.strftime("%Y-%m-%d %H:%M:%S"),
                                reader_name,
                                gender,
                                selected_dict["name"],
                                state["current_set"],
                                wav_filename,
                                converted_wav_filename,
                                final_id,
                                final_sim,
                                state["all_words"],
                                state["detail_logs"],
                            )

                            # --- 次セット初期化 ---
                            state["current_set"] += 1
                            state["scores"] = {}
                            state["seen_words"] = {}
                            state["all_words"] = []
                            state["detail_logs"] = []
                            state["result_text"] = ""

                            state["is_speaking"] = True
                            state["silence_start_time"] = None
                            state["shimonoku_finished"] = False
                            state["uonoku_start_time"] = None
                            state["recorded_frames"] = [data]
                            state["converted_recorded_frames"] = [
                                processed_audio.tobytes()
                            ]

                            rec.Reset()

                            if state["current_set"] > MAX_SETS:
                                break
                            continue

                # --- 認識・加点処理 (上の句2.0秒以内) ---
                if (
                    state["shimonoku_finished"]
                    and state["uonoku_start_time"] is not None
                ):
                    elapsed_uonoku = time.time() - state["uonoku_start_time"]

                    if elapsed_uonoku <= LIMIT_UONOKU_SECONDS:
                        int16_data = (
                            (processed_audio * 32768.0)
                            .astype(np.int16)
                            .tobytes()
                        )
                        if not rec.AcceptWaveform(int16_data):
                            state["result_text"] = json.loads(
                                rec.PartialResult()
                            ).get("partial", "")
                        else:
                            state["result_text"] = json.loads(
                                rec.Result()
                            ).get("text", "")

                        if state["result_text"]:
                            new_words = state["result_text"].split()
                            for word in new_words:
                                if word in tag_map:
                                    for tid in tag_map[word]:
                                        state["seen_words"].setdefault(tid, set())
                                        if word not in state["seen_words"][tid]:
                                            state["scores"][tid] = state[
                                                "scores"
                                            ].get(tid, 0) + len(word)
                                            state["seen_words"][tid].add(word)
                                            if word not in state["all_words"]:
                                                state["all_words"].append(word)

                                            current_sim = (
                                                state["scores"][tid]
                                                / max_scores.get(tid, 1)
                                            ) * 100
                                            state["detail_logs"].append(
                                                [
                                                    datetime.now().strftime(
                                                        "%H:%M:%S.%f"
                                                    )[:-3],
                                                    word,
                                                    tid,
                                                    state["scores"][tid],
                                                    max_scores.get(tid),
                                                    f"{current_sim:.1f}%",
                                                ]
                                            )

    except KeyboardInterrupt:
        print("\nユーザーによって処理が中断されました。")
    finally:
        state["is_running"] = False
        print("\n=== 全セットの自動稼働を終了しました ===")


if __name__ == "__main__":
    run_realtime_karuta()

In [ ]:
import csv
import json
import os
import queue
import sqlite3
import sys
import threading
import time
import wave
from datetime import datetime

from IPython.display import clear_output
import numpy as np
import sounddevice as sd

# ピッチ変換ライブラリ (scipy)
from scipy.signal import resample
from vosk import KaldiRecognizer, Model

# ==============================================================================
# --- 1. 定数・辞書定義・初期設定 ---
# ==============================================================================

DICTIONARIES = {
    "1": {
        "name": "ひらがな版（統一文字数辞書）",
        "path": "data/karuta_hiragana_aligned.csv",
        "type": "aligned_hiragana",
    },
    "2": {
        "name": "カタカナ版（統一文字数辞書）",
        "path": "data/karuta_katakana_aligned.csv",
        "type": "aligned_katakana",
    },
    "3": {
        "name": "決まり字・手動追加版（サマリー辞書）",
        "path": "data/karuta_card_summary_by_kimari_len.csv",
        "type": "summary",
    },
}

RESULTS_CSV = "results/recognition_history.csv"
TEXT_LOG_DIR = "text_logs"
DB_PATH = "results/karuta_results.db"
SAMPLERATE = 16000

# 音量・時間制御パラメタ
VOLUME_THRESHOLD = 0.01  # 音声を感知するしきい値
WAIT_SECONDS = 0.8  # 下の句終了から上の句移行までの静寂時間
START_SET = 1  # 開始セット
MAX_SETS = 102  # 最大セット数

LIMIT_UONOKU_SECONDS = 2.0  # 上の句加点処理を実行する秒数
UONOKU_MIN_SECONDS = 11.0  # 次セットへの移行ブロック時間


# ==============================================================================
# --- 2. データベース制御 (SQLite) ---
# ==============================================================================
def init_db():
    """DBテーブルの初期化およびデータモデルに合わせたカラム定義"""
    os.makedirs(os.path.dirname(DB_PATH), exist_ok=True)
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    # 1. メインのセット結果テーブル
    cursor.execute(
        """
    CREATE TABLE IF NOT EXISTS recognition_sets (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        timestamp TEXT NOT NULL,
        reader_name TEXT,
        gender TEXT,
        dict_name TEXT,
        set_num INTEGER NOT NULL,
        wav_filename TEXT,
        audio_file_path TEXT,
        converted_wav_filename TEXT,
        pitch_shifted_file_path TEXT,
        final_id TEXT,
        final_similarity REAL,
        recognized_words TEXT,
        recognized_text TEXT
    )
    """
    )

    # 既存DBテーブルのスキーマ自動補正 (カラムの動的追加)
    cursor.execute("PRAGMA table_info(recognition_sets)")
    columns = [column[1] for column in cursor.fetchall()]
    
    migration_columns = {
        "converted_wav_filename": "TEXT",
        "audio_file_path": "TEXT",
        "pitch_shifted_file_path": "TEXT",
        "recognized_text": "TEXT"
    }
    
    for col_name, col_type in migration_columns.items():
        if col_name not in columns:
            cursor.execute(f"ALTER TABLE recognition_sets ADD COLUMN {col_name} {col_type}")

    # 2. 詳細加点ログテーブル
    cursor.execute(
        """
    CREATE TABLE IF NOT EXISTS recognition_details (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        set_id INTEGER NOT NULL,
        log_time TEXT,
        word TEXT,
        card_id TEXT,
        current_score INTEGER,
        max_score INTEGER,
        similarity REAL,
        score_gained INTEGER,
        rank INTEGER,
        FOREIGN KEY (set_id) REFERENCES recognition_sets (id) ON DELETE CASCADE
    )
    """
    )
    conn.commit()
    conn.close()


def save_to_db(
    timestamp,
    reader_name,
    gender,
    dict_name,
    set_num,
    wav_filename,
    audio_file_path,
    converted_wav_filename,
    pitch_shifted_file_path,
    final_id,
    final_sim,
    all_words,
    detail_logs,
):
    """結果をSQLite DBに保存"""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    recognized_words_str = " ".join(all_words)

    cursor.execute(
        """
    INSERT INTO recognition_sets (
        timestamp, reader_name, gender, dict_name, set_num, 
        wav_filename, audio_file_path, converted_wav_filename, pitch_shifted_file_path, 
        final_id, final_similarity, recognized_words, recognized_text
    ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """,
        (
            timestamp,
            reader_name,
            gender,
            dict_name,
            set_num,
            wav_filename,
            audio_file_path,
            converted_wav_filename,
            pitch_shifted_file_path,
            final_id,
            final_sim,
            recognized_words_str,
            recognized_words_str,  # recognized_text にも同様に格納
        ),
    )

    set_id = cursor.lastrowid

    # 詳細ログの保存
    for log in detail_logs:
        # logの構成: [log_time, word, card_id, current_score, max_score, sim_str, gain, rank]
        sim_val = float(str(log[5]).replace("%", ""))
        score_gained = log[6] if len(log) > 6 else len(log[1])
        rank = log[7] if len(log) > 7 else None

        cursor.execute(
            """
        INSERT INTO recognition_details (
            set_id, log_time, word, card_id, current_score, max_score, similarity, score_gained, rank
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
        """,
            (set_id, log[0], log[1], log[2], log[3], log[4], sim_val, score_gained, rank),
        )

    conn.commit()
    conn.close()


# ==============================================================================
# --- 3. 音声ピッチ変換ロジック ---
# ==============================================================================
def pitch_shift(audio_data, semitones=4.0):
    """簡易リサンプルによるピッチシフト処理"""
    if semitones == 0:
        return audio_data

    factor = 2.0 ** (semitones / 12.0)
    orig_len = len(audio_data)

    resampled = resample(audio_data, int(orig_len / factor))

    if len(resampled) < orig_len:
        resampled = np.pad(resampled, (0, orig_len - len(resampled)), "constant")
    else:
        resampled = resampled[:orig_len]

    return resampled.astype(np.float32)


# ==============================================================================
# --- 4. 辞書読み込み & 初期化 ---
# ==============================================================================

def prepare_karuta_system(dict_info):
    file_path = dict_info["path"]
    dict_type = dict_info["type"]

    if not os.path.exists(file_path):
        print(f"Error: {file_path} が見つかりません。パスを確認してください。")
        return None

    grammar_set = set()
    tag_map = {}
    card_max_scores = {}

    with open(file_path, "r", encoding="utf-8-sig") as f:
        reader = csv.reader(f)
        header = next(reader, None)

        for row in reader:
            if not row:
                continue

            num = row[0].strip()
            if num in ["0", "00", ""]:
                continue

            registered_words = []

            # --- 1 & 2. ひらがな・カタカナ版辞書 ---
            if dict_type in ["aligned_hiragana", "aligned_katakana"]:
                if len(row) < 6:
                    continue
                raw_words = row[5].strip()
                if raw_words:
                    registered_words = raw_words.split()

                denominator_seen = set(registered_words)
                try:
                    fixed_max_char_score = int(row[4].strip())
                except ValueError:
                    fixed_max_char_score = sum(
                        len(w) for w in denominator_seen
                    )

            # --- 3. サマリー辞書 ---
            elif dict_type == "summary":
                if len(row) < 14:
                    continue
                # 7列目〜13列目 (インデックス 7〜13) の単語をループで回収
                for col_idx in range(7, 14):
                    if row[col_idx].strip():
                        registered_words.extend(row[col_idx].strip().split())

                denominator_seen = set(registered_words)
                fixed_max_char_score = sum(len(w) for w in denominator_seen)

            else:
                continue

            # 単語が何も登録されていない行はスキップ
            if not registered_words:
                continue

            card_max_scores[num] = fixed_max_char_score

            for w in denominator_seen:
                grammar_set.add(w)
                tag_map.setdefault(w, []).append(num)

    print(f"\n[システム初期化完了 - {dict_info['name']}]")
    print(f" ➔ 対象ファイル: {file_path}")
    print(f" ➔ Vosk登録総語数: {len(grammar_set)}語")
    print(f" ➔ 登録札数: {len(card_max_scores)}枚")

    return {
        "grammar": list(grammar_set),
        "tag_map": tag_map,
        "max_scores": card_max_scores,
    }

# ==============================================================================
# --- 5. ダッシュボード表示 & ログ保存 ---
# ==============================================================================
def print_dashboard(
    scores,
    max_scores,
    partial_text,
    model_name,
    current_set,
    total_sets,
    detail_logs,
    status_mode,
    elapsed,
    current_volume,
    pause_time,
    reader_name,
    gender,
    dict_name,
):
    clear_output(wait=True)
    results = []
    for tid, score in scores.items():
        max_s = max_scores.get(tid, 1)
        sim = (score / max_s) * 100
        results.append({"id": tid, "score": score, "max": max_s, "sim": sim})
    top_5 = sorted(results, key=lambda x: x["sim"], reverse=True)[:5]

    if status_mode == "wait":
        status_label = "【発声待ち】音源（前の札の下の句）の再生を待っています..."
    elif status_mode == "skip":
        if pause_time > 0:
            status_label = f"【下の句 ➔ 静寂検知中】⏱ {pause_time:.1f}秒 / {WAIT_SECONDS}秒で上の句判定へ (加点無効)"
        else:
            status_label = "【前の札の下の句を検出中...】 音圧検出中 (加点無効)"
    elif status_mode == "recognizing":
        rem = max(0.0, LIMIT_UONOKU_SECONDS - elapsed)
        status_label = f"【上の句 認識実行中】 ⏱ 認識残り: {rem:.1f}秒 / {LIMIT_UONOKU_SECONDS}秒"
    else:
        rem_next = max(0.0, UONOKU_MIN_SECONDS - elapsed)
        status_label = f"【認識完了 ➔ 次セット待機中】 ⏱ 次の発声許可まで残り: {rem_next:.1f}秒 (認識停止中)"

    output = [
        "=" * 80,
        f"読み手: {reader_name} ({gender}) | 辞書: {dict_name}",
        f"設定: [Model: {model_name}] [進行: {current_set} / {total_sets} セット]",
        f"状況: {status_label}",
        f"音量: ［ {current_volume:.4f} ］ (しきい値: {VOLUME_THRESHOLD})",
        "=" * 80,
    ]

    clean_partial = partial_text.replace("\n", " ")
    display_partial = (
        clean_partial[-50:] if len(clean_partial) > 50 else clean_partial
    )
    output.append(f" | 認識中の言葉  | {display_partial.ljust(50)}")
    output.extend(
        [
            "-" * 80,
            f"{'順位':<4}｜{'推定札':<10} | {'加点/満点':<12}｜{'類似度':<8} ｜棒グラフ",
            "-" * 80,
        ]
    )
    for i in range(5):
        if i < len(top_5):
            res = top_5[i]
            bar_count = int(res["sim"] / 5)
            bar = "■" * bar_count + "□" * (20 - bar_count)
            output.append(
                f"{i+1}位 ｜ {res['id']:>3}番      | {res['score']:>3} / {res['max']:>3} pts | {res['sim']:>5.1f}%    ｜|{bar}"
            )
        else:
            output.append(
                f"{i+1}位 ｜ ---番      |    0 / --- pts |    0.0%    ｜|{'□'*20}"
            )

    output.extend(["-" * 80, " 【リアルタイム認識ログ（直近5件）】", "-" * 80])
    if detail_logs:
        recent_logs = detail_logs[-5:][::-1]
        for log in recent_logs:
            output.append(
                f" [{log[0]}] 単語: {log[1]:<8} ➔ {log[2]:>3}番に加点 (+{len(log[1])}pts)"
            )
    else:
        output.append(" まだキーワードが認識されていません。")

    output.extend(["=" * 80])
    print("\n".join(output))


def save_to_result_csv(data):
    os.makedirs(os.path.dirname(RESULTS_CSV), exist_ok=True)
    file_exists = os.path.isfile(RESULTS_CSV)
    with open(RESULTS_CSV, "a", encoding="utf-8-sig", newline="") as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow(
                [
                    "日時",
                    "読み手",
                    "性別",
                    "生音声ファイル名",
                    "変換音声ファイル名",
                    "最終推定ID",
                    "最終類似度",
                    "認識単語",
                ]
            )
        writer.writerow(data)


# ==============================================================================
# --- 6. メイン実行関数 ---
# ==============================================================================
def run_realtime_karuta():
    init_db()  # DBのテーブル初期設定

    print("=========================================")
    print("    かるた音声認識システム 設定")
    print("=========================================")

    # 1. 読み手名の入力
    print("\n【1】 読み手の名前")
    reader_name = input("【読手入力】 読み手の名前を入力してください: ").strip()
    if not reader_name:
        reader_name = "Unknown"

    # 2. 性別の選択
    print("\n【2】 読み手の性別を選択してください:")
    print("  1: 男性 (ピッチ高音化補正: +4半音)")
    print("  2: 女性 (ピッチ変更なし)")
    g_choice = input("【性別選択】 番号を入力してください (1 or 2): ").strip()

    if g_choice == "1":
        gender = "男性"
        pitch_shift_semitones = 4.0
    else:
        gender = "女性"
        pitch_shift_semitones = 0.0

    # 3. 辞書の選択
    print("\n【3】 使用する辞書を選択してください:")
    for k, v in DICTIONARIES.items():
        print(f"  {k}: {v['name']}")
    d_choice = input("【辞書選択】 番号を入力してください (1〜3): ").strip()
    selected_dict = DICTIONARIES.get(d_choice, DICTIONARIES["1"])

    # 4. モデル選択
    print("\n【4】 使用モデルを選択してください:")
    print("  1: Standard (精度重視) / 2: Small (速度重視)")
    m_choice = input("【モデル選択】 番号を入力してください (1 or 2): ").strip()
    model_name = "model_standard" if m_choice == "1" else "model_small"

    # システム・辞書準備
    sys_data = prepare_karuta_system(selected_dict)
    if not sys_data:
        return

    grammar = sys_data["grammar"]
    tag_map = sys_data["tag_map"]
    max_scores = sys_data["max_scores"]

    print(f"\n--- Vosk モデル読み込み中 ({model_name}) ---")
    model = Model(model_name)
    rec = KaldiRecognizer(model, SAMPLERATE, json.dumps(grammar, ensure_ascii=False))

    q = queue.Queue()

    def callback(indata, frames, time_info, status):
        q.put(bytes(indata))

    state = {
        "current_set": START_SET,
        "scores": {},
        "seen_words": {},
        "all_words": [],
        "detail_logs": [],
        "result_text": "",
        "current_volume": 0.0,
        "is_speaking": False,
        "silence_start_time": None,
        "shimonoku_finished": False,
        "uonoku_start_time": None,
        "status_mode": "wait",
        "elapsed": 0.0,
        "pause_time": 0.0,
        "is_running": True,
        "recorded_frames": [],
        "converted_recorded_frames": [],  # ピッチ変換後音声保持用
    }

    def dashboard_refresh_worker():
        while state["is_running"]:
            if not state["shimonoku_finished"]:
                if state["is_speaking"]:
                    state["status_mode"] = "skip"
                    if state["silence_start_time"] is not None:
                        state["pause_time"] = (
                            time.time() - state["silence_start_time"]
                        )
                    else:
                        state["pause_time"] = 0.0
                else:
                    state["status_mode"] = "wait"
                    state["pause_time"] = 0.0
                state["elapsed"] = 0.0
            else:
                if state["uonoku_start_time"] is not None:
                    state["elapsed"] = time.time() - state["uonoku_start_time"]
                state["pause_time"] = 0.0
                if state["elapsed"] <= LIMIT_UONOKU_SECONDS:
                    state["status_mode"] = "recognizing"
                else:
                    state["status_mode"] = "waiting_next"

            print_dashboard(
                state["scores"],
                max_scores,
                state["result_text"],
                model_name,
                state["current_set"],
                MAX_SETS,
                state["detail_logs"],
                state["status_mode"],
                state["elapsed"],
                state["current_volume"],
                state["pause_time"],
                reader_name,
                gender,
                selected_dict["name"],
            )
            time.sleep(0.1)

    refresh_thread = threading.Thread(target=dashboard_refresh_worker, daemon=True)
    refresh_thread.start()

    print("\n>>> 音声ストリームの受付を開始します。")
    try:
        with sd.RawInputStream(
            samplerate=SAMPLERATE,
            blocksize=1600,
            channels=1,
            dtype="float32",
            callback=callback,
        ):
            while state["current_set"] <= MAX_SETS and state["is_running"]:
                data = q.get()
                state["recorded_frames"].append(data)

                audio_data = np.frombuffer(data, dtype=np.float32)
                if len(audio_data) == 0:
                    continue

                # ピッチ補正（男性声の場合はシフト）
                processed_audio = pitch_shift(audio_data, pitch_shift_semitones)
                
                # 変換済みのデータフレームをバイナリとして保持
                state["converted_recorded_frames"].append(processed_audio.tobytes())

                state["current_volume"] = np.sqrt(np.mean(processed_audio**2))

                # --- 状態管理 ---
                if not state["shimonoku_finished"]:
                    if state["current_volume"] > VOLUME_THRESHOLD:
                        state["is_speaking"] = True
                        state["silence_start_time"] = None
                    else:
                        if state["is_speaking"]:
                            if state["silence_start_time"] is None:
                                state["silence_start_time"] = time.time()
                            elif (
                                time.time() - state["silence_start_time"]
                                >= WAIT_SECONDS
                            ):
                                state["is_speaking"] = False
                                state["shimonoku_finished"] = True
                                state["uonoku_start_time"] = time.time()
                                state["silence_start_time"] = None
                                rec.Reset()
                                state["result_text"] = ""
                else:
                    elapsed_uonoku = time.time() - state["uonoku_start_time"]

                    # 次セットへの移行判定 (11秒以降に次の声を感知)
                    if state["current_volume"] > VOLUME_THRESHOLD:
                        if elapsed_uonoku > UONOKU_MIN_SECONDS:
                            final_id, final_sim = "---", 0.0
                            if state["scores"]:
                                final_id = max(
                                    state["scores"],
                                    key=lambda k: state["scores"][k]
                                    / max_scores.get(k, 1),
                                )
                                final_sim = (
                                    state["scores"][final_id]
                                    / max_scores.get(final_id, 1)
                                ) * 100

                            dt = datetime.now()
                            dt_str = dt.strftime("%Y%m%d_%H%M%S")
                            base_name = f"set_{state['current_set']:03d}_{dt_str}"
                            wav_filename = f"{base_name}.wav"
                            converted_wav_filename = f"{base_name}_converted.wav"

                            # 1-A. 生WAVの保存
                            os.makedirs("recordings", exist_ok=True)
                            full_path_wav = os.path.join("recordings", wav_filename)
                            with wave.open(full_path_wav, "wb") as wf:
                                wf.setnchannels(1)
                                wf.setsampwidth(2)
                                wf.setframerate(SAMPLERATE)
                                all_audio = np.frombuffer(
                                    b"".join(state["recorded_frames"]),
                                    dtype=np.float32,
                                )
                                all_audio_int16 = (
                                    (all_audio * 32768.0)
                                    .astype(np.int16)
                                    .tobytes()
                                )
                                wf.writeframes(all_audio_int16)

                            # 1-B. 変換後WAVの保存
                            os.makedirs("converted_recordings", exist_ok=True)
                            full_path_converted = os.path.join(
                                "converted_recordings", converted_wav_filename
                            )
                            with wave.open(full_path_converted, "wb") as wf:
                                wf.setnchannels(1)
                                wf.setsampwidth(2)
                                wf.setframerate(SAMPLERATE)
                                all_converted_audio = np.frombuffer(
                                    b"".join(state["converted_recorded_frames"]),
                                    dtype=np.float32,
                                )
                                converted_int16 = (
                                    (all_converted_audio * 32768.0)
                                    .astype(np.int16)
                                    .tobytes()
                                )
                                wf.writeframes(converted_int16)

                            # 2. 全体CSV保存
                            save_to_result_csv(
                                [
                                    dt.strftime("%Y-%m-%d %H:%M:%S"),
                                    reader_name,
                                    gender,
                                    wav_filename,
                                    converted_wav_filename,
                                    final_id,
                                    f"{final_sim:.1f}%",
                                    " ".join(state["all_words"]),
                                ]
                            )

                            # 3. DBへ保存 (新スキーマに完全準拠)
                            save_to_db(
                                dt.strftime("%Y-%m-%d %H:%M:%S"),
                                reader_name,
                                gender,
                                selected_dict["name"],
                                state["current_set"],
                                wav_filename,
                                full_path_wav,
                                converted_wav_filename,
                                full_path_converted,
                                final_id,
                                final_sim,
                                state["all_words"],
                                state["detail_logs"],
                            )

                            # --- 次セット初期化 ---
                            state["current_set"] += 1
                            state["scores"] = {}
                            state["seen_words"] = {}
                            state["all_words"] = []
                            state["detail_logs"] = []
                            state["result_text"] = ""

                            state["is_speaking"] = True
                            state["silence_start_time"] = None
                            state["shimonoku_finished"] = False
                            state["uonoku_start_time"] = None
                            state["recorded_frames"] = [data]
                            state["converted_recorded_frames"] = [
                                processed_audio.tobytes()
                            ]

                            rec.Reset()

                            if state["current_set"] > MAX_SETS:
                                break
                            continue

                # --- 認識・加点処理 (上の句2.0秒以内) ---
                if (
                    state["shimonoku_finished"]
                    and state["uonoku_start_time"] is not None
                ):
                    elapsed_uonoku = time.time() - state["uonoku_start_time"]

                    if elapsed_uonoku <= LIMIT_UONOKU_SECONDS:
                        int16_data = (
                            (processed_audio * 32768.0)
                            .astype(np.int16)
                            .tobytes()
                        )
                        if not rec.AcceptWaveform(int16_data):
                            state["result_text"] = json.loads(
                                rec.PartialResult()
                            ).get("partial", "")
                        else:
                            state["result_text"] = json.loads(
                                rec.Result()
                            ).get("text", "")

                        if state["result_text"]:
                            new_words = state["result_text"].split()
                            for word in new_words:
                                if word in tag_map:
                                    for tid in tag_map[word]:
                                        state["seen_words"].setdefault(tid, set())
                                        if word not in state["seen_words"][tid]:
                                            gain = len(word)
                                            state["scores"][tid] = state[
                                                "scores"
                                            ].get(tid, 0) + gain
                                            state["seen_words"][tid].add(word)
                                            if word not in state["all_words"]:
                                                state["all_words"].append(word)

                                            current_sim = (
                                                state["scores"][tid]
                                                / max_scores.get(tid, 1)
                                            ) * 100
                                            
                                            # 順位計算 (計算時のスコアに基づく)
                                            ranked = sorted(
                                                state["scores"].items(),
                                                key=lambda x: x[1] / max_scores.get(x[0], 1),
                                                reverse=True,
                                            )
                                            rank = next(
                                                (i + 1 for i, r in enumerate(ranked) if r[0] == tid),
                                                None,
                                            )

                                            state["detail_logs"].append(
                                                [
                                                    datetime.now().strftime(
                                                        "%H:%M:%S.%f"
                                                    )[:-3],
                                                    word,
                                                    tid,
                                                    state["scores"][tid],
                                                    max_scores.get(tid),
                                                    f"{current_sim:.1f}%",
                                                    gain,
                                                    rank,
                                                ]
                                            )

    except KeyboardInterrupt:
        print("\nユーザーによって処理が中断されました。")
    finally:
        state["is_running"] = False
        print("\n=== 全セットの自動稼働を終了しました ===")


if __name__ == "__main__":
    run_realtime_karuta()

In [ ]:
import csv
import json
import os
import queue
import sqlite3
import sys
import threading
import time
import wave
from datetime import datetime

from IPython.display import clear_output
import numpy as np
import sounddevice as sd
from vosk import KaldiRecognizer, Model

# ==============================================================================
# --- 1. 定数・辞書定義・初期設定 ---
# ==============================================================================

DICTIONARIES = {
    "1": {
        "name": "ひらがな版（統一文字数辞書）",
        "path": "data/karuta_hiragana_aligned.csv",
        "type": "aligned_hiragana",
    },
    "2": {
        "name": "カタカナ版（統一文字数辞書）",
        "path": "data/karuta_katakana_aligned.csv",
        "type": "aligned_katakana",
    },
    "3": {
        "name": "決まり字・手動追加版（サマリー辞書）",
        "path": "data/karuta_card_summary_by_kimari_len.csv",
        "type": "summary",
    },
}

RESULTS_CSV = "results/recognition_history.csv"
TEXT_LOG_DIR = "text_logs"
DB_PATH = "results/karuta_results.db"
SAMPLERATE = 16000

# 音量・時間制御パラメタ
VOLUME_THRESHOLD = 0.01  # 音声を感知するしきい値
WAIT_SECONDS = 0.8  # 下の句終了から上の句移行までの静寂時間
START_SET = 1  # 開始セット
MAX_SETS = 102  # 最大セット数

LIMIT_UONOKU_SECONDS = 2.0  # 上の句加点処理を実行する秒数
UONOKU_MIN_SECONDS = 11.0  # 次セットへの移行ブロック時間


# ==============================================================================
# --- 2. データベース制御 (SQLite) - 既存DB構造（session_id）に準拠 ---
# ==============================================================================
def init_db():
    """DBテーブルの初期化およびデータモデルに合わせたカラム定義"""
    os.makedirs(os.path.dirname(DB_PATH), exist_ok=True)
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    # 1. recognition_sets テーブル
    cursor.execute(
        """
    CREATE TABLE IF NOT EXISTS recognition_sets (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        timestamp TEXT NOT NULL,
        reader_name TEXT,
        gender TEXT,
        dict_type TEXT,
        set_num INTEGER NOT NULL,
        wav_filename TEXT,
        audio_file_path TEXT,
        converted_wav_filename TEXT,
        pitch_shifted_file_path TEXT,
        final_id TEXT,
        final_similarity REAL,
        recognized_words TEXT,
        recognized_text TEXT
    )
    """
    )

    cursor.execute("PRAGMA table_info(recognition_sets)")
    sets_columns = [column[1] for column in cursor.fetchall()]

    sets_migration_columns = {
        "dict_type": "TEXT",
        "converted_wav_filename": "TEXT",
        "audio_file_path": "TEXT",
        "pitch_shifted_file_path": "TEXT",
        "recognized_text": "TEXT",
    }

    for col_name, col_type in sets_migration_columns.items():
        if col_name not in sets_columns:
            cursor.execute(
                f"ALTER TABLE recognition_sets ADD COLUMN {col_name} {col_type}"
            )

    # 2. recognition_details テーブル (既存DB構造に準拠)
    cursor.execute(
        """
    CREATE TABLE IF NOT EXISTS recognition_details (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        set_id INTEGER NOT NULL,
        log_time TEXT,
        word TEXT,
        card_id TEXT,
        current_score INTEGER,
        max_score INTEGER,
        similarity REAL,
        session_id INTEGER,
        FOREIGN KEY (set_id) REFERENCES recognition_sets (id) ON DELETE CASCADE
    )
    """
    )

    conn.commit()
    conn.close()


def save_to_db(
    timestamp,
    reader_name,
    gender,
    dict_type,
    set_num,
    wav_filename,
    audio_file_path,
    converted_wav_filename,
    pitch_shifted_file_path,
    final_id,
    final_sim,
    all_words,
    detail_logs,
):
    """結果をSQLite DBに保存 (既存DBの構造に合わせて保存)"""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    recognized_words_str = " ".join(all_words)

    cursor.execute(
        """
    INSERT INTO recognition_sets (
        timestamp, reader_name, gender, dict_type, set_num, 
        wav_filename, audio_file_path, converted_wav_filename, pitch_shifted_file_path, 
        final_id, final_similarity, recognized_words, recognized_text
    ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """,
        (
            timestamp,
            reader_name,
            gender,
            dict_type,
            set_num,
            wav_filename,
            audio_file_path,
            converted_wav_filename,
            pitch_shifted_file_path,
            final_id,
            final_sim,
            recognized_words_str,
            recognized_words_str,
        ),
    )

    set_id = cursor.lastrowid

    # 既存DBのカラム構造 (set_id, log_time, word, card_id, current_score, max_score, similarity, session_id) に保存
    for log in detail_logs:
        sim_val = float(str(log[5]).replace("%", ""))

        cursor.execute(
            """
        INSERT INTO recognition_details (
            set_id, log_time, word, card_id, current_score, max_score, similarity, session_id
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?)
        """,
            (
                set_id,
                log[0],  # log_time
                log[1],  # word
                log[2],  # card_id
                log[3],  # current_score
                log[4],  # max_score
                sim_val,  # similarity
                set_num,  # session_id として set_num を挿入
            ),
        )

    conn.commit()
    conn.close()


# ==============================================================================
# --- 3. 滑らかなリアルタイム音声ピッチ変換クラス ---
# ==============================================================================
class SmoothPitchShifter:
    """内部リングバッファを用いた滑らかなピッチシフト処理クラス"""

    def __init__(self, semitones=4.0, buffer_size=6400, overlap_size=640):
        self.semitones = semitones
        self.factor = 2.0 ** (semitones / 12.0)
        self.buffer_size = buffer_size
        self.overlap_size = overlap_size

        self.input_buffer = np.zeros(self.buffer_size, dtype=np.float32)
        self.prev_tail = np.zeros(self.overlap_size, dtype=np.float32)

        if self.overlap_size > 0:
            window = np.hanning(self.overlap_size * 2)
            self.fade_in = window[: self.overlap_size]
            self.fade_out = window[self.overlap_size :]

    def process(self, chunk):
        if self.semitones == 0:
            return chunk

        chunk_len = len(chunk)

        self.input_buffer = np.roll(self.input_buffer, -chunk_len)
        self.input_buffer[-chunk_len:] = chunk

        x_orig = np.arange(self.buffer_size)
        x_resampled = np.arange(0, self.buffer_size, self.factor)
        resampled_buf = np.interp(x_resampled, x_orig, self.input_buffer).astype(
            np.float32
        )

        out_chunk = resampled_buf[-chunk_len:].copy()

        if self.overlap_size > 0 and chunk_len >= self.overlap_size:
            out_chunk[: self.overlap_size] = (
                out_chunk[: self.overlap_size] * self.fade_in
                + self.prev_tail * self.fade_out
            )
            self.prev_tail = out_chunk[-self.overlap_size :].copy()

        return out_chunk


# ==============================================================================
# --- 4. 辞書読み込み & 初期化 ---
# ==============================================================================
def prepare_karuta_system(dict_info):
    file_path = dict_info["path"]
    dict_type = dict_info["type"]

    if not os.path.exists(file_path):
        print(f"Error: {file_path} が見つかりません。パスを確認してください。")
        return None

    grammar_set = set()
    tag_map = {}
    card_max_scores = {}

    with open(file_path, "r", encoding="utf-8-sig") as f:
        reader = csv.reader(f)
        header = next(reader, None)

        for row in reader:
            if not row:
                continue

            num = row[0].strip()
            if num in ["0", "00", ""]:
                continue

            registered_words = []

            if dict_type in ["aligned_hiragana", "aligned_katakana"]:
                if len(row) < 6:
                    continue
                raw_words = row[5].strip()
                if raw_words:
                    registered_words = raw_words.split()

                denominator_seen = set(registered_words)
                try:
                    fixed_max_char_score = int(row[4].strip())
                except ValueError:
                    fixed_max_char_score = sum(
                        len(w) for w in denominator_seen
                    )

            elif dict_type == "summary":
                if len(row) < 14:
                    continue
                for col_idx in range(7, 14):
                    if row[col_idx].strip():
                        registered_words.extend(
                            row[col_idx].strip().split()
                        )

                denominator_seen = set(registered_words)
                fixed_max_char_score = sum(len(w) for w in denominator_seen)

            else:
                continue

            if not registered_words:
                continue

            card_max_scores[num] = fixed_max_char_score

            for w in denominator_seen:
                grammar_set.add(w)
                tag_map.setdefault(w, []).append(num)

    print(f"\n[システム初期化完了 - {dict_info['name']}]")
    print(f" ➔ 対象ファイル: {file_path}")
    print(f" ➔ Vosk登録総語数: {len(grammar_set)}語")
    print(f" ➔ 登録札数: {len(card_max_scores)}枚")

    return {
        "grammar": list(grammar_set),
        "tag_map": tag_map,
        "max_scores": card_max_scores,
    }


# ==============================================================================
# --- 5. ダッシュボード表示 & ログ保存 ---
# ==============================================================================
def print_dashboard(
    scores,
    max_scores,
    partial_text,
    model_name,
    current_set,
    total_sets,
    detail_logs,
    status_mode,
    elapsed,
    current_volume,
    pause_time,
    reader_name,
    gender,
    dict_name,
):
    clear_output(wait=True)
    results = []
    for tid, score in scores.items():
        max_s = max_scores.get(tid, 1)
        sim = (score / max_s) * 100
        results.append({"id": tid, "score": score, "max": max_s, "sim": sim})
    top_5 = sorted(results, key=lambda x: x["sim"], reverse=True)[:5]

    if status_mode == "wait":
        status_label = "【発声待ち】音源（前の札の下の句）の再生を待っています..."
    elif status_mode == "skip":
        if pause_time > 0:
            status_label = f"【下の句 ➔ 静寂検知中】⏱ {pause_time:.1f}秒 / {WAIT_SECONDS}秒で上の句判定へ (加点無効)"
        else:
            status_label = "【前の札の下の句を検出中...】 音圧検出中 (加点無効)"
    elif status_mode == "recognizing":
        rem = max(0.0, LIMIT_UONOKU_SECONDS - elapsed)
        status_label = f"【上の句 認識実行中】 ⏱ 認識残り: {rem:.1f}秒 / {LIMIT_UONOKU_SECONDS}秒"
    else:
        rem_next = max(0.0, UONOKU_MIN_SECONDS - elapsed)
        status_label = f"【認識完了 ➔ 次セット待機中】 ⏱ 次の発声許可まで残り: {rem_next:.1f}秒 (認識停止中)"

    output = [
        "=" * 80,
        f"読み手: {reader_name} ({gender}) | 辞書: {dict_name}",
        f"設定: [Model: {model_name}] [進行: {current_set} / {total_sets} セット]",
        f"状況: {status_label}",
        f"音量: ［ {current_volume:.4f} ］ (しきい値: {VOLUME_THRESHOLD})",
        "=" * 80,
    ]

    clean_partial = partial_text.replace("\n", " ")
    display_partial = (
        clean_partial[-50:] if len(clean_partial) > 50 else clean_partial
    )
    output.append(f" | 認識中の言葉  | {display_partial.ljust(50)}")
    output.extend(
        [
            "-" * 80,
            f"{'順位':<4}｜{'推定札':<10} | {'加点/満点':<12}｜{'類似度':<8} ｜棒グラフ",
            "-" * 80,
        ]
    )
    for i in range(5):
        if i < len(top_5):
            res = top_5[i]
            bar_count = int(res["sim"] / 5)
            bar = "■" * bar_count + "□" * (20 - bar_count)
            output.append(
                f"{i+1}位 ｜ {res['id']:>3}番      | {res['score']:>3} / {res['max']:>3} pts | {res['sim']:>5.1f}%    ｜|{bar}"
            )
        else:
            output.append(
                f"{i+1}位 ｜ ---番      |    0 / --- pts |    0.0%    ｜|{'□'*20}"
            )

    output.extend(["-" * 80, " 【リアルタイム認識ログ（直近5件）】", "-" * 80])
    if detail_logs:
        recent_logs = detail_logs[-5:][::-1]
        for log in recent_logs:
            output.append(
                f" [{log[0]}] 単語: {log[1]:<8} ➔ {log[2]:>3}番に加点 (+{len(log[1])}pts)"
            )
    else:
        output.append(" まだキーワードが認識されていません。")

    output.extend(["=" * 80])
    print("\n".join(output))


def save_to_result_csv(data):
    os.makedirs(os.path.dirname(RESULTS_CSV), exist_ok=True)
    file_exists = os.path.isfile(RESULTS_CSV)
    with open(RESULTS_CSV, "a", encoding="utf-8-sig", newline="") as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow(
                [
                    "日時",
                    "読み手",
                    "性別",
                    "生音声ファイル名",
                    "変換音声ファイル名",
                    "最終推定ID",
                    "最終類似度",
                    "認識単語",
                ]
            )
        writer.writerow(data)


# ==============================================================================
# --- 6. メイン実行関数 ---
# ==============================================================================
def run_realtime_karuta():
    init_db()

    print("=========================================")
    print("    かるた音声認識システム 設定")
    print("=========================================")

    # 1. 読み手名の入力
    print("\n【1】 読み手の名前")
    reader_name = input("【読手入力】 読み手の名前を入力してください: ").strip()
    if not reader_name:
        reader_name = "Unknown"

    # 2. 性別の選択
    print("\n【2】 読み手の性別を選択してください:")
    print("  1: 男性 (ピッチ高音化補正: +4半音)")
    print("  2: 女性 (ピッチ変更なし)")
    g_choice = input("【性別選択】 番号を入力してください (1 or 2): ").strip()

    if g_choice == "1":
        gender = "男性"
        pitch_shift_semitones = 4.0
    else:
        gender = "女性"
        pitch_shift_semitones = 0.0

    # 3. 辞書の選択
    print("\n【3】 使用する辞書を選択してください:")
    for k, v in DICTIONARIES.items():
        print(f"  {k}: {v['name']}")
    d_choice = input("【辞書選択】 番号を入力してください (1〜3): ").strip()
    selected_dict = DICTIONARIES.get(d_choice, DICTIONARIES["1"])

    # 4. モデル選択
    print("\n【4】 使用モデルを選択してください:")
    print("  1: Standard (精度重視) / 2: Small (速度重視)")
    m_choice = input("【モデル選択】 番号を入力してください (1 or 2): ").strip()
    model_name = "model_standard" if m_choice == "1" else "model_small"

    # システム・辞書準備
    sys_data = prepare_karuta_system(selected_dict)
    if not sys_data:
        return

    grammar = sys_data["grammar"]
    tag_map = sys_data["tag_map"]
    max_scores = sys_data["max_scores"]

    print(f"\n--- Vosk モデル読み込み中 ({model_name}) ---")
    model = Model(model_name)
    rec = KaldiRecognizer(model, SAMPLERATE, json.dumps(grammar, ensure_ascii=False))

    pitch_shifter = SmoothPitchShifter(
        semitones=pitch_shift_semitones, buffer_size=6400, overlap_size=640
    )

    q = queue.Queue()

    def callback(indata, frames, time_info, status):
        q.put(bytes(indata))

    state = {
        "current_set": START_SET,
        "scores": {},
        "seen_words": {},
        "all_words": [],
        "detail_logs": [],
        "result_text": "",
        "current_volume": 0.0,
        "is_speaking": False,
        "silence_start_time": None,
        "shimonoku_finished": False,
        "uonoku_start_time": None,
        "status_mode": "wait",
        "elapsed": 0.0,
        "pause_time": 0.0,
        "is_running": True,
        "recorded_frames": [],
        "converted_recorded_frames": [],
    }

    def dashboard_refresh_worker():
        while state["is_running"]:
            if not state["shimonoku_finished"]:
                if state["is_speaking"]:
                    state["status_mode"] = "skip"
                    if state["silence_start_time"] is not None:
                        state["pause_time"] = (
                            time.time() - state["silence_start_time"]
                        )
                    else:
                        state["pause_time"] = 0.0
                else:
                    state["status_mode"] = "wait"
                    state["pause_time"] = 0.0
                state["elapsed"] = 0.0
            else:
                if state["uonoku_start_time"] is not None:
                    state["elapsed"] = time.time() - state["uonoku_start_time"]
                state["pause_time"] = 0.0
                if state["elapsed"] <= LIMIT_UONOKU_SECONDS:
                    state["status_mode"] = "recognizing"
                else:
                    state["status_mode"] = "waiting_next"

            print_dashboard(
                state["scores"],
                max_scores,
                state["result_text"],
                model_name,
                state["current_set"],
                MAX_SETS,
                state["detail_logs"],
                state["status_mode"],
                state["elapsed"],
                state["current_volume"],
                state["pause_time"],
                reader_name,
                gender,
                selected_dict["name"],
            )
            time.sleep(0.1)

    refresh_thread = threading.Thread(target=dashboard_refresh_worker, daemon=True)
    refresh_thread.start()

    print("\n>>> 音声ストリームの受付を開始します。")
    try:
        with sd.RawInputStream(
            samplerate=SAMPLERATE,
            blocksize=1600,
            channels=1,
            dtype="float32",
            callback=callback,
        ):
            while state["current_set"] <= MAX_SETS and state["is_running"]:
                data = q.get()
                state["recorded_frames"].append(data)

                audio_data = np.frombuffer(data, dtype=np.float32)
                if len(audio_data) == 0:
                    continue

                processed_audio = pitch_shifter.process(audio_data)

                state["converted_recorded_frames"].append(processed_audio.tobytes())
                state["current_volume"] = np.sqrt(np.mean(processed_audio**2))

                # --- 状態管理 ---
                if not state["shimonoku_finished"]:
                    if state["current_volume"] > VOLUME_THRESHOLD:
                        state["is_speaking"] = True
                        state["silence_start_time"] = None
                    else:
                        if state["is_speaking"]:
                            if state["silence_start_time"] is None:
                                state["silence_start_time"] = time.time()
                            elif (
                                time.time() - state["silence_start_time"]
                                >= WAIT_SECONDS
                            ):
                                state["is_speaking"] = False
                                state["shimonoku_finished"] = True
                                state["uonoku_start_time"] = time.time()
                                state["silence_start_time"] = None
                                rec.Reset()
                                state["result_text"] = ""
                else:
                    elapsed_uonoku = time.time() - state["uonoku_start_time"]

                    if state["current_volume"] > VOLUME_THRESHOLD:
                        if elapsed_uonoku > UONOKU_MIN_SECONDS:
                            final_id, final_sim = "---", 0.0
                            if state["scores"]:
                                final_id = max(
                                    state["scores"],
                                    key=lambda k: state["scores"][k]
                                    / max_scores.get(k, 1),
                                )
                                final_sim = (
                                    state["scores"][final_id]
                                    / max_scores.get(final_id, 1)
                                ) * 100

                            dt = datetime.now()
                            dt_str = dt.strftime("%Y%m%d_%H%M%S")
                            base_name = f"set_{state['current_set']:03d}_{dt_str}"
                            wav_filename = f"{base_name}.wav"
                            converted_wav_filename = f"{base_name}_converted.wav"

                            os.makedirs("recordings", exist_ok=True)
                            full_path_wav = os.path.join("recordings", wav_filename)
                            with wave.open(full_path_wav, "wb") as wf:
                                wf.setnchannels(1)
                                wf.setsampwidth(2)
                                wf.setframerate(SAMPLERATE)
                                all_audio = np.frombuffer(
                                    b"".join(state["recorded_frames"]),
                                    dtype=np.float32,
                                )
                                all_audio_int16 = (
                                    (all_audio * 32768.0)
                                    .astype(np.int16)
                                    .tobytes()
                                )
                                wf.writeframes(all_audio_int16)

                            os.makedirs("converted_recordings", exist_ok=True)
                            full_path_converted = os.path.join(
                                "converted_recordings", converted_wav_filename
                            )
                            with wave.open(full_path_converted, "wb") as wf:
                                wf.setnchannels(1)
                                wf.setsampwidth(2)
                                wf.setframerate(SAMPLERATE)
                                all_converted_audio = np.frombuffer(
                                    b"".join(state["converted_recorded_frames"]),
                                    dtype=np.float32,
                                )
                                converted_int16 = (
                                    (all_converted_audio * 32768.0)
                                    .astype(np.int16)
                                    .tobytes()
                                )
                                wf.writeframes(converted_int16)

                            save_to_result_csv(
                                [
                                    dt.strftime("%Y-%m-%d %H:%M:%S"),
                                    reader_name,
                                    gender,
                                    wav_filename,
                                    converted_wav_filename,
                                    final_id,
                                    f"{final_sim:.1f}%",
                                    " ".join(state["all_words"]),
                                ]
                            )

                            save_to_db(
                                dt.strftime("%Y-%m-%d %H:%M:%S"),
                                reader_name,
                                gender,
                                selected_dict["type"],
                                state["current_set"],
                                wav_filename,
                                full_path_wav,
                                converted_wav_filename,
                                full_path_converted,
                                final_id,
                                final_sim,
                                state["all_words"],
                                state["detail_logs"],
                            )

                            state["current_set"] += 1
                            state["scores"] = {}
                            state["seen_words"] = {}
                            state["all_words"] = []
                            state["detail_logs"] = []
                            state["result_text"] = ""

                            state["is_speaking"] = True
                            state["silence_start_time"] = None
                            state["shimonoku_finished"] = False
                            state["uonoku_start_time"] = None
                            state["recorded_frames"] = [data]
                            state["converted_recorded_frames"] = [
                                processed_audio.tobytes()
                            ]

                            rec.Reset()

                            if state["current_set"] > MAX_SETS:
                                break
                            continue

                # --- 認識・加点処理 (上の句2.0秒以内) ---
                if (
                    state["shimonoku_finished"]
                    and state["uonoku_start_time"] is not None
                ):
                    elapsed_uonoku = time.time() - state["uonoku_start_time"]

                    if elapsed_uonoku <= LIMIT_UONOKU_SECONDS:
                        int16_data = (
                            (processed_audio * 32768.0)
                            .astype(np.int16)
                            .tobytes()
                        )
                        if not rec.AcceptWaveform(int16_data):
                            state["result_text"] = json.loads(
                                rec.PartialResult()
                            ).get("partial", "")
                        else:
                            state["result_text"] = json.loads(
                                rec.Result()
                            ).get("text", "")

                        if state["result_text"]:
                            new_words = state["result_text"].split()
                            for word in new_words:
                                if word in tag_map:
                                    for tid in tag_map[word]:
                                        state["seen_words"].setdefault(tid, set())
                                        if word not in state["seen_words"][tid]:
                                            gain = len(word)
                                            state["scores"][tid] = state[
                                                "scores"
                                            ].get(tid, 0) + gain
                                            state["seen_words"][tid].add(word)
                                            if word not in state["all_words"]:
                                                state["all_words"].append(word)

                                            current_sim = (
                                                state["scores"][tid]
                                                / max_scores.get(tid, 1)
                                            ) * 100

                                            state["detail_logs"].append(
                                                [
                                                    datetime.now().strftime(
                                                        "%H:%M:%S.%f"
                                                    )[:-3],
                                                    word,
                                                    tid,
                                                    state["scores"][tid],
                                                    max_scores.get(tid),
                                                    f"{current_sim:.1f}%",
                                                ]
                                            )

    except KeyboardInterrupt:
        print("\nユーザーによって処理が中断されました。")
    finally:
        state["is_running"] = False
        print("\n=== 全セットの自動稼働を終了しました ===")


if __name__ == "__main__":
    run_realtime_karuta()

詳細CSV復活

In [ ]:
import csv
import json
import os
import queue
import sqlite3
import sys
import threading
import time
import wave
from datetime import datetime

import numpy as np
import sounddevice as sd
from IPython.display import clear_output
from vosk import KaldiRecognizer, Model

# ==============================================================================
# --- 1. 定数・辞書定義・初期設定 ---
# ==============================================================================

DICTIONARIES = {
    "1": {
        "name": "ひらがな版（統一文字数辞書）",
        "path": "data/karuta_hiragana_aligned.csv",
        "type": "aligned_hiragana",
    },
    "2": {
        "name": "カタカナ版（統一文字数辞書）",
        "path": "data/karuta_katakana_aligned.csv",
        "type": "aligned_katakana",
    },
    "3": {
        "name": "決まり字・手動追加版（サマリー辞書）",
        "path": "data/karuta_card_summary_by_kimari_len.csv",
        "type": "summary",
    },
}

RESULTS_CSV = "results/recognition_history.csv"
TEXT_LOG_DIR = "text_logs"
DB_PATH = "results/karuta_results.db"
SAMPLERATE = 16000

# 音量・時間制御パラメタ
VOLUME_THRESHOLD = 0.01  # 音声を感知するしきい値
WAIT_SECONDS = 0.8  # 下の句終了から上の句移行までの静寂時間
START_SET = 1  # 開始セット
MAX_SETS = 102  # 最大セット数

LIMIT_UONOKU_SECONDS = 2.0  # 上の句加点処理を実行する秒数
UONOKU_MIN_SECONDS = 11.0  # 次セットへの移行ブロック時間


# ==============================================================================
# --- 2. データベース制御 (SQLite) ---
# ==============================================================================
def init_db():
    """DBテーブルの初期化およびデータモデルに合わせたカラム定義"""
    os.makedirs(os.path.dirname(DB_PATH), exist_ok=True)
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    cursor.execute(
        """
    CREATE TABLE IF NOT EXISTS recognition_sets (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        timestamp TEXT NOT NULL,
        reader_name TEXT,
        gender TEXT,
        dict_type TEXT,
        set_num INTEGER NOT NULL,
        wav_filename TEXT,
        audio_file_path TEXT,
        converted_wav_filename TEXT,
        pitch_shifted_file_path TEXT,
        final_id TEXT,
        final_similarity REAL,
        recognized_words TEXT,
        recognized_text TEXT
    )
    """
    )

    cursor.execute("PRAGMA table_info(recognition_sets)")
    sets_columns = [column[1] for column in cursor.fetchall()]

    sets_migration_columns = {
        "dict_type": "TEXT",
        "converted_wav_filename": "TEXT",
        "audio_file_path": "TEXT",
        "pitch_shifted_file_path": "TEXT",
        "recognized_text": "TEXT",
    }

    for col_name, col_type in sets_migration_columns.items():
        if col_name not in sets_columns:
            cursor.execute(
                f"ALTER TABLE recognition_sets ADD COLUMN {col_name} {col_type}"
            )

    cursor.execute(
        """
    CREATE TABLE IF NOT EXISTS recognition_details (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        set_id INTEGER NOT NULL,
        log_time TEXT,
        word TEXT,
        card_id TEXT,
        current_score INTEGER,
        max_score INTEGER,
        similarity REAL,
        session_id INTEGER,
        FOREIGN KEY (set_id) REFERENCES recognition_sets (id) ON DELETE CASCADE
    )
    """
    )

    conn.commit()
    conn.close()


def save_to_db(
    timestamp,
    reader_name,
    gender,
    dict_type,
    set_num,
    wav_filename,
    audio_file_path,
    converted_wav_filename,
    pitch_shifted_file_path,
    final_id,
    final_sim,
    all_words,
    detail_logs,
):
    """結果をSQLite DBに保存"""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    recognized_words_str = " ".join(all_words)

    cursor.execute(
        """
    INSERT INTO recognition_sets (
        timestamp, reader_name, gender, dict_type, set_num, 
        wav_filename, audio_file_path, converted_wav_filename, pitch_shifted_file_path, 
        final_id, final_similarity, recognized_words, recognized_text
    ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """,
        (
            timestamp,
            reader_name,
            gender,
            dict_type,
            set_num,
            wav_filename,
            audio_file_path,
            converted_wav_filename,
            pitch_shifted_file_path,
            final_id,
            final_sim,
            recognized_words_str,
            recognized_words_str,
        ),
    )

    set_id = cursor.lastrowid

    for log in detail_logs:
        sim_val = float(str(log[5]).replace("%", ""))

        cursor.execute(
            """
        INSERT INTO recognition_details (
            set_id, log_time, word, card_id, current_score, max_score, similarity, session_id
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?)
        """,
            (
                set_id,
                log[0],  # log_time
                log[1],  # word
                log[2],  # card_id
                log[3],  # current_score
                log[4],  # max_score
                sim_val, # similarity
                set_num, # session_id
            ),
        )

    conn.commit()
    conn.close()


# ==============================================================================
# --- 3. 滑らかなリアルタイム音声ピッチ変換クラス ---
# ==============================================================================
class SmoothPitchShifter:
    """内部リングバッファを用いた滑らかなピッチシフト処理クラス"""

    def __init__(self, semitones=4.0, buffer_size=6400, overlap_size=640):
        self.semitones = semitones
        self.factor = 2.0 ** (semitones / 12.0)
        self.buffer_size = buffer_size
        self.overlap_size = overlap_size

        self.input_buffer = np.zeros(self.buffer_size, dtype=np.float32)
        self.prev_tail = np.zeros(self.overlap_size, dtype=np.float32)

        if self.overlap_size > 0:
            window = np.hanning(self.overlap_size * 2)
            self.fade_in = window[: self.overlap_size]
            self.fade_out = window[self.overlap_size :]

    def process(self, chunk):
        if self.semitones == 0:
            return chunk

        chunk_len = len(chunk)

        self.input_buffer = np.roll(self.input_buffer, -chunk_len)
        self.input_buffer[-chunk_len:] = chunk

        x_orig = np.arange(self.buffer_size)
        x_resampled = np.arange(0, self.buffer_size, self.factor)
        resampled_buf = np.interp(x_resampled, x_orig, self.input_buffer).astype(
            np.float32
        )

        out_chunk = resampled_buf[-chunk_len:].copy()

        if self.overlap_size > 0 and chunk_len >= self.overlap_size:
            out_chunk[: self.overlap_size] = (
                out_chunk[: self.overlap_size] * self.fade_in
                + self.prev_tail * self.fade_out
            )
            self.prev_tail = out_chunk[-self.overlap_size :].copy()

        return out_chunk


# ==============================================================================
# --- 4. 辞書読み込み & 初期化 ---
# ==============================================================================
def prepare_karuta_system(dict_info):
    file_path = dict_info["path"]
    dict_type = dict_info["type"]

    if not os.path.exists(file_path):
        print(f"Error: {file_path} が見つかりません。パスを確認してください。")
        return None

    grammar_set = set()
    tag_map = {}
    card_max_scores = {}

    with open(file_path, "r", encoding="utf-8-sig") as f:
        reader = csv.reader(f)
        header = next(reader, None)

        for row in reader:
            if not row:
                continue

            num = row[0].strip()
            if num in ["0", "00", ""]:
                continue

            registered_words = []

            if dict_type in ["aligned_hiragana", "aligned_katakana"]:
                if len(row) < 6:
                    continue
                raw_words = row[5].strip()
                if raw_words:
                    registered_words = raw_words.split()

                denominator_seen = set(registered_words)
                try:
                    fixed_max_char_score = int(row[4].strip())
                except ValueError:
                    fixed_max_char_score = sum(
                        len(w) for w in denominator_seen
                    )

            elif dict_type == "summary":
                if len(row) < 14:
                    continue
                for col_idx in range(7, 14):
                    if row[col_idx].strip():
                        registered_words.extend(
                            row[col_idx].strip().split()
                        )

                denominator_seen = set(registered_words)
                fixed_max_char_score = sum(len(w) for w in denominator_seen)

            else:
                continue

            if not registered_words:
                continue

            card_max_scores[num] = fixed_max_char_score

            for w in denominator_seen:
                grammar_set.add(w)
                tag_map.setdefault(w, []).append(num)

    print(f"\n[システム初期化完了 - {dict_info['name']}]")
    print(f" ➔ 対象ファイル: {file_path}")
    print(f" ➔ Vosk登録総語数: {len(grammar_set)}語")
    print(f" ➔ 登録札数: {len(card_max_scores)}枚")

    return {
        "grammar": list(grammar_set),
        "tag_map": tag_map,
        "max_scores": card_max_scores,
    }


# ==============================================================================
# --- 5. ダッシュボード表示 & ファイルログ保存 ---
# ==============================================================================
def print_dashboard(
    scores,
    max_scores,
    partial_text,
    model_name,
    current_set,
    total_sets,
    detail_logs,
    status_mode,
    elapsed,
    current_volume,
    pause_time,
    reader_name,
    gender,
    dict_name,
):
    clear_output(wait=True)
    results = []
    for tid, score in scores.items():
        max_s = max_scores.get(tid, 1)
        sim = (score / max_s) * 100
        results.append({"id": tid, "score": score, "max": max_s, "sim": sim})
    top_5 = sorted(results, key=lambda x: x["sim"], reverse=True)[:5]

    if status_mode == "wait":
        status_label = "【発声待ち】音源（前の札の下の句）の再生を待っています..."
    elif status_mode == "skip":
        if pause_time > 0:
            status_label = f"【下の句 ➔ 静寂検知中】⏱ {pause_time:.1f}秒 / {WAIT_SECONDS}秒で上の句判定へ (加点無効)"
        else:
            status_label = "【前の札の下の句を検出中...】 音圧検出中 (加点無効)"
    elif status_mode == "recognizing":
        rem = max(0.0, LIMIT_UONOKU_SECONDS - elapsed)
        status_label = f"【上の句 認識実行中】 ⏱ 認識残り: {rem:.1f}秒 / {LIMIT_UONOKU_SECONDS}秒"
    else:
        rem_next = max(0.0, UONOKU_MIN_SECONDS - elapsed)
        status_label = f"【認識完了 ➔ 次セット待機中】 ⏱ 次の発声許可まで残り: {rem_next:.1f}秒 (認識停止中)"

    output = [
        "=" * 80,
        f"読み手: {reader_name} ({gender}) | 辞書: {dict_name}",
        f"設定: [Model: {model_name}] [進行: {current_set} / {total_sets} セット]",
        f"状況: {status_label}",
        f"音量: ［ {current_volume:.4f} ］ (しきい値: {VOLUME_THRESHOLD})",
        "=" * 80,
    ]

    clean_partial = partial_text.replace("\n", " ")
    display_partial = (
        clean_partial[-50:] if len(clean_partial) > 50 else clean_partial
    )
    output.append(f" | 認識中の言葉  | {display_partial.ljust(50)}")
    output.extend(
        [
            "-" * 80,
            f"{'順位':<4}｜{'推定札':<10} | {'加点/満点':<12}｜{'類似度':<8} ｜棒グラフ",
            "-" * 80,
        ]
    )
    for i in range(5):
        if i < len(top_5):
            res = top_5[i]
            bar_count = int(res["sim"] / 5)
            bar = "■" * bar_count + "□" * (20 - bar_count)
            output.append(
                f"{i+1}位 ｜ {res['id']:>3}番      | {res['score']:>3} / {res['max']:>3} pts | {res['sim']:>5.1f}%    ｜|{bar}"
            )
        else:
            output.append(
                f"{i+1}位 ｜ ---番      |    0 / --- pts |    0.0%    ｜|{'□'*20}"
            )

    output.extend(["-" * 80, " 【リアルタイム認識ログ（直近5件）】", "-" * 80])
    if detail_logs:
        recent_logs = detail_logs[-5:][::-1]
        for log in recent_logs:
            output.append(
                f" [{log[0]}] 単語: {log[1]:<8} ➔ {log[2]:>3}番に加点 (+{len(log[1])}pts)"
            )
    else:
        output.append(" まだキーワードが認識されていません。")

    output.extend(["=" * 80])
    print("\n".join(output))


def save_to_result_csv(data):
    """全体履歴CSV（results/recognition_history.csv）への追記保存"""
    os.makedirs(os.path.dirname(RESULTS_CSV), exist_ok=True)
    file_exists = os.path.isfile(RESULTS_CSV)
    with open(RESULTS_CSV, "a", encoding="utf-8-sig", newline="") as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow(
                [
                    "日時",
                    "読み手",
                    "性別",
                    "生音声ファイル名",
                    "変換音声ファイル名",
                    "最終推定ID",
                    "最終類似度",
                    "認識単語",
                ]
            )
        writer.writerow(data)


def save_set_detail_csv(file_path, detail_logs):
    """セットごとの時系列詳細ログ（results/set_XXX_YYYYMMDD_HHMMSS.csv）の保存"""
    os.makedirs(os.path.dirname(file_path), exist_ok=True)
    with open(file_path, "w", encoding="utf-8-sig", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            "時刻",
            "認識単語",
            "加点札ID",
            "現在スコア",
            "満点スコア",
            "類似度",
        ])
        writer.writerows(detail_logs)


def save_to_text_file(set_num, dict_name, base_name, final_id, final_sim, all_words, detail_logs):
    """セットごとの解析結果テキストファイルの保存"""
    os.makedirs(TEXT_LOG_DIR, exist_ok=True)
    txt_path = os.path.join(TEXT_LOG_DIR, f"{base_name}.txt")

    with open(txt_path, "w", encoding="utf-8") as f:
        f.write(f"=== 第 {set_num} セット 解析結果 (2秒判定モデル) ===\n")
        f.write(f"使用辞書: {dict_name}\n")
        f.write(f"日時: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"識別ファイルベース名: {base_name}\n")
        f.write(f"最終推定札ID: {final_id}\n")
        f.write(f"最終類似度: {final_sim:.1f}%\n")
        f.write(f"認識に成功した全単語群: {' '.join(all_words)}\n")
        f.write("\n--- 時系列詳細加点ログ ---\n")
        for log in detail_logs:
            f.write(f"[{log[0]}] 単語: {log[1]:<8} ➔ {log[2]}番に加点 (+{len(log[1])}pts) | 現在累計: {log[3]}/{log[4]} ({log[5]})\n")


# ==============================================================================
# --- 6. メイン実行関数 ---
# ==============================================================================
def run_realtime_karuta():
    init_db()

    print("=========================================")
    print("    かるた音声認識システム 設定")
    print("=========================================")

    # 1. 読み手名の入力
    reader_name = input("【1】 読み手の名前を入力してください: ").strip()
    if not reader_name:
        reader_name = "Unknown"

    # 2. 性別の選択
    gender_input = input("【2】 読み手の性別を選択してください (1: 男性 / 2: 女性 / 3: その他): ").strip()
    gender_map = {"1": "男性", "2": "女性", "3": "その他"}
    gender = gender_map.get(gender_input, "不明")

    # 3. 辞書の選択
    print("\n【3】 使用する辞書を選択してください:")
    for key, dict_info in DICTIONARIES.items():
        print(f"  {key}: {dict_info['name']}")
    dict_choice = input("選択 (1-3) -> ").strip()
    selected_dict = DICTIONARIES.get(dict_choice, DICTIONARIES["1"])

    # 4. モデルの選択
    print("\n【4】 Voskモデルを選択してください:")
    print("  1: Standard (精度重視)")
    print("  2: Small (速度重視)")
    model_choice = input("選択 (1-2) -> ").strip()
    model_name = "model_standard" if model_choice == "1" else "model_small"

    # ピッチシフトの設定（女性ボイスを男性寄りに調整する場合などのオプション）
    pitch_shifter = SmoothPitchShifter(semitones=0.0)

    # 辞書データの準備
    sys_data = prepare_karuta_system(selected_dict)
    if not sys_data:
        return

    grammar = sys_data["grammar"]
    tag_map = sys_data["tag_map"]
    max_scores = sys_data["max_scores"]

    print(f"\n--- Vosk モデル読み込み中 ({model_name}) ---")
    try:
        model = Model(model_name)
    except Exception as e:
        print(f"Error: Voskモデルの読み込みに失敗しました ({e})")
        return

    rec = KaldiRecognizer(model, SAMPLERATE, json.dumps(grammar, ensure_ascii=False))

    q = queue.Queue()

    def callback(indata, frames, time_info, status):
        q.put(bytes(indata))

    state = {
        "current_set": START_SET,
        "scores": {},
        "seen_words": {},
        "all_words": [],
        "detail_logs": [],
        "result_text": "",
        "current_volume": 0.0,
        "is_speaking": False,
        "silence_start_time": None,
        "shimonoku_finished": False,
        "uonoku_start_time": None,
        "status_mode": "wait",
        "elapsed": 0.0,
        "pause_time": 0.0,
        "is_running": True,
        "recorded_frames": [],
        "pitch_shifted_frames": [],
    }

    def dashboard_refresh_worker():
        while state["is_running"]:
            if not state["shimonoku_finished"]:
                if state["is_speaking"]:
                    state["status_mode"] = "skip"
                    if state["silence_start_time"] is not None:
                        state["pause_time"] = time.time() - state["silence_start_time"]
                    else:
                        state["pause_time"] = 0.0
                else:
                    state["status_mode"] = "wait"
                    state["pause_time"] = 0.0
                state["elapsed"] = 0.0
            else:
                if state["uonoku_start_time"] is not None:
                    state["elapsed"] = time.time() - state["uonoku_start_time"]
                state["pause_time"] = 0.0
                if state["elapsed"] <= LIMIT_UONOKU_SECONDS:
                    state["status_mode"] = "recognizing"
                else:
                    state["status_mode"] = "waiting_next"

            print_dashboard(
                state["scores"],
                max_scores,
                state["result_text"],
                model_name,
                state["current_set"],
                MAX_SETS,
                state["detail_logs"],
                state["status_mode"],
                state["elapsed"],
                state["current_volume"],
                state["pause_time"],
                reader_name,
                gender,
                selected_dict["name"],
            )
            time.sleep(0.1)

    refresh_thread = threading.Thread(target=dashboard_refresh_worker, daemon=True)
    refresh_thread.start()

    print("\n>>> 音声ストリームの受付を開始します。")
    try:
        with sd.RawInputStream(
            samplerate=SAMPLERATE, blocksize=1600, channels=1, dtype="float32", callback=callback
        ):
            while state["current_set"] <= MAX_SETS and state["is_running"]:
                data = q.get()
                state["recorded_frames"].append(data)

                audio_data = np.frombuffer(data, dtype=np.float32)
                if len(audio_data) == 0:
                    continue

                # リアルタイムピッチシフトの適応
                processed_audio = pitch_shifter.process(audio_data)
                state["pitch_shifted_frames"].append(processed_audio.tobytes())

                state["current_volume"] = np.sqrt(np.mean(audio_data**2))

                # --- 状態管理（下の句・上の句・次セット移行の判定） ---
                if not state["shimonoku_finished"]:
                    if state["current_volume"] > VOLUME_THRESHOLD:
                        state["is_speaking"] = True
                        state["silence_start_time"] = None
                    else:
                        if state["is_speaking"]:
                            if state["silence_start_time"] is None:
                                state["silence_start_time"] = time.time()
                            elif time.time() - state["silence_start_time"] >= WAIT_SECONDS:
                                state["is_speaking"] = False
                                state["shimonoku_finished"] = True
                                state["uonoku_start_time"] = time.time()
                                state["silence_start_time"] = None

                                rec.Reset()
                                state["result_text"] = ""
                else:
                    elapsed_uonoku = time.time() - state["uonoku_start_time"]

                    # 上の句認識終了後、一定時間経過して音量を検出したら次のセットへ
                    if state["current_volume"] > VOLUME_THRESHOLD:
                        if elapsed_uonoku > UONOKU_MIN_SECONDS:
                            # --- 現セットのスコア集計 ---
                            final_id, final_sim = "---", 0.0
                            if state["scores"]:
                                final_id = max(
                                    state["scores"],
                                    key=lambda k: state["scores"][k] / max_scores.get(k, 1),
                                )
                                final_sim = (
                                    state["scores"][final_id] / max_scores.get(final_id, 1)
                                ) * 100

                            dt = datetime.now()
                            dt_str = dt.strftime("%Y%m%d_%H%M%S")
                            base_name = f"set_{state['current_set']:03d}_{dt_str}"

                            raw_wav_name = f"{base_name}_raw.wav"
                            converted_wav_name = f"{base_name}_pitch.wav"
                            detail_csv_name = f"{base_name}.csv"

                            full_path_raw_wav = os.path.join("recordings", raw_wav_name)
                            full_path_converted_wav = os.path.join("recordings", converted_wav_name)
                            full_path_detail_csv = os.path.join("results", detail_csv_name)

                            os.makedirs("recordings", exist_ok=True)
                            os.makedirs("results", exist_ok=True)

                            # 1. Raw Audio WAV保存
                            with wave.open(full_path_raw_wav, "wb") as wf:
                                wf.setnchannels(1)
                                wf.setsampwidth(2)
                                wf.setframerate(SAMPLERATE)
                                all_audio = np.frombuffer(
                                    b"".join(state["recorded_frames"]), dtype=np.float32
                                )
                                all_audio_int16 = (all_audio * 32768.0).astype(np.int16).tobytes()
                                wf.writeframes(all_audio_int16)

                            # 2. Pitch-shifted WAV保存
                            with wave.open(full_path_converted_wav, "wb") as wf:
                                wf.setnchannels(1)
                                wf.setsampwidth(2)
                                wf.setframerate(SAMPLERATE)
                                all_pitch_audio = np.frombuffer(
                                    b"".join(state["pitch_shifted_frames"]), dtype=np.float32
                                )
                                all_pitch_int16 = (all_pitch_audio * 32768.0).astype(np.int16).tobytes()
                                wf.writeframes(all_pitch_int16)

                            # 3. 詳細ログCSVファイル保存 (要件の追加機能)
                            save_set_detail_csv(full_path_detail_csv, state["detail_logs"])

                            # 4. 全体サマリーCSV履歴保存
                            save_to_result_csv([
                                dt.strftime("%Y-%m-%d %H:%M:%S"),
                                reader_name,
                                gender,
                                raw_wav_name,
                                converted_wav_name,
                                final_id,
                                f"{final_sim:.1f}%",
                                " ".join(state["all_words"]),
                            ])

                            # 5. SQLite データベース保存
                            save_to_db(
                                dt.strftime("%Y-%m-%d %H:%M:%S"),
                                reader_name,
                                gender,
                                selected_dict["name"],
                                state["current_set"],
                                raw_wav_name,
                                full_path_raw_wav,
                                converted_wav_name,
                                full_path_converted_wav,
                                final_id,
                                final_sim,
                                state["all_words"],
                                state["detail_logs"],
                            )

                            # 6. テキスト形式ログの保存
                            save_to_text_file(
                                state["current_set"],
                                selected_dict["name"],
                                base_name,
                                final_id,
                                final_sim,
                                state["all_words"],
                                state["detail_logs"],
                            )

                            # --- 次セット用データの初期化 ---
                            state["current_set"] += 1
                            state["scores"] = {}
                            state["seen_words"] = {}
                            state["all_words"] = []
                            state["detail_logs"] = []
                            state["result_text"] = ""

                            state["is_speaking"] = True
                            state["silence_start_time"] = None
                            state["shimonoku_finished"] = False
                            state["uonoku_start_time"] = None
                            state["recorded_frames"] = [data]
                            state["pitch_shifted_frames"] = [processed_audio.tobytes()]

                            rec.Reset()

                            if state["current_set"] > MAX_SETS:
                                break
                            continue

                # --- 音声認識・加点処理 (上の句開始から2.0秒以内のみ実行) ---
                if state["shimonoku_finished"] and state["uonoku_start_time"] is not None:
                    elapsed_uonoku = time.time() - state["uonoku_start_time"]

                    if elapsed_uonoku <= LIMIT_UONOKU_SECONDS:
                        int16_data = (processed_audio * 32768.0).astype(np.int16).tobytes()
                        if not rec.AcceptWaveform(int16_data):
                            state["result_text"] = json.loads(rec.PartialResult()).get(
                                "partial", ""
                            )
                        else:
                            state["result_text"] = json.loads(rec.Result()).get("text", "")

                        if state["result_text"]:
                            new_words = state["result_text"].split()
                            for word in new_words:
                                if word in tag_map:
                                    for tid in tag_map[word]:
                                        state["seen_words"].setdefault(tid, set())
                                        if word not in state["seen_words"][tid]:
                                            state["scores"][tid] = state["scores"].get(
                                                tid, 0
                                            ) + len(word)
                                            state["seen_words"][tid].add(word)
                                            if word not in state["all_words"]:
                                                state["all_words"].append(word)

                                            current_sim = (
                                                state["scores"][tid] / max_scores.get(tid, 1)
                                            ) * 100
                                            state["detail_logs"].append([
                                                datetime.now().strftime("%H:%M:%S.%f")[:-3],
                                                word,
                                                tid,
                                                state["scores"][tid],
                                                max_scores.get(tid),
                                                f"{current_sim:.1f}%",
                                            ])

    except KeyboardInterrupt:
        print("\nユーザーによって処理が中断されました。")
    finally:
        state["is_running"] = False
        print("\n=== 全セットの自動稼働を終了しました ===")


if __name__ == "__main__":
    run_realtime_karuta()